# 🚀 DAQathon Session 1: Advanced Machine Learning

This notebook picks up where `session1_machine_learning.ipynb` leaves off. It uses the same dataset setup, then goes deeper on the questions that matter once a baseline model works:

- tuning hyperparameters on the validation split, and checking whether the gains hold on test,
- giving tree models more temporal context and trying other tree families,
- changing the target definition itself,
- predicting one QC flag per timestamp with a deeper CNN,
- comparing loss functions for rare labels,
- and a forecasting transformer that learns normal behaviour and flags large prediction errors.

Work through the machine-learning notebook first: this notebook does not repeat the introductions to Random Forests, CNNs, or Transformers. The data preparation is explained in `session1_data_preparation.ipynb`.

This notebook includes hyperparameter searches, so it takes longer than the machine-learning notebook. Start with `DATA_FRACTION = 0.1`.

**What to edit:** code cells that start with `# ✏️ EDIT` hold the settings you are meant to change, and the heading above them starts with ✏️ too. Every other code cell can be run as-is.

Notebook outline:

1. **Dataset and cache setup.** The same options as the machine-learning notebook.
2. **Random Forest.** Hyperparameter search, temporal context features, tree families, and target design.
3. **CNN.** Hyperparameter search and a per-timestamp sequence-labelling CNN.
4. **Loss functions.** Train the same CNN with each loss option and compare the rare-class results.
5. **Transformer.** Forecast good data, then flag large prediction errors as anomalies.
6. **Reflection.** Decide which changes are worth carrying forward.

## 🧰 One-Time Cluster Setup

Before you use these notebooks on Narval or FIR for the first time, open a cluster terminal and run:

```bash
/project/def-kmoran/shared/daqathon/envs/daqathon-ml-venv/bin/jupyter kernelspec install --user /project/def-kmoran/shared/daqathon/kernels/daqathon-ml
```

This is a **one-time per-user step**. After that:

- refresh JupyterHub if it was already open,
- open the notebook from your cloned repo,
- select the shared `Daqathon ML` kernel.

## Import Packages And Helper Functions

Run the next cell once before choosing a dataset setup option. It prepares the Python environment by importing plotting/data libraries, scikit-learn tools, and the DAQathon helper functions used later in the model sections.

After the imports are ready, choose **Option A** for a prepared workshop cache, **Option B** to create a cache from your own CSV files, or **Option C** to load an existing custom cache.

In [ ]:
# Standard-library imports used throughout the notebook.
from pathlib import Path
import copy
import json
import os
import pickle
import subprocess
import sys
from time import perf_counter

# Find the repo root before importing utilities from scripts/.
# This lets the notebook run from either the repo root or the notebooks/ folder.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

# Data, plotting, and machine-learning packages.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Session setup helpers.
from scripts.session1_resume_utils import load_ml_section_state, show_setup_json

# Shared modelling helpers used by the RF, k-means, CNN, and Transformer sections.
from scripts.session1_modeling import (
    add_temporal_context_features,
    add_indicator_channels,
    add_tabular_baseline_features,
    apply_target_strategy,
    build_cache_bundle_paths,
    build_labeled_intervals,
    build_model_frame,
    build_reviewed_target_frame,
    build_sequence_split_bundle,
    build_sequence_split_bundle_from_frames,
    build_sequence_label_interval_data,
    build_window_classification_interval_data,
    clean_source_file_label,
    compute_contiguous_split_target_distribution,
    compute_interval_classification_metrics,
    compute_split_share_gap,
    build_classification_loss,
    describe_indicator_channels,
    describe_time_range_selection,
    DEFAULT_FLAG_PALETTE,
    evaluate_classifier,
    fit_extra_trees,
    fit_kmeans,
    fit_random_forest,
    infer_interval_origin,
    load_full_row_level_frame,
    load_rows_for_time_range,
    load_selected_row_level_frame,
    materialize_reviewed_split_frames,
    merge_adjacent_intervals,
    per_class_recall,
    plot_cluster_window_examples,
    plot_flag_examples,
    plot_time_series_with_bands,
    predict_cnn_window_model,
    predict_sequence_label_cnn,
    predict_transformer_sequence_model,
    predict_transformer_window_model,
    resolve_cache_bundle_paths,
    resolve_runtime_output_root,
    report_average,
    run_cnn_search,
    run_cnn_search_from_frames,
    run_rf_search,
    sample_frame_by_strategy,
    scan_interleaved_block_rows,
    select_time_range,
    split_frame_by_strategy,
    stage_cache_into_runtime,
    stage_directory_into_runtime,
    summarize_split_distributions,
    summarize_target_by_time_bin,
    time_gap_segment_ids,
    SUPPORTED_SPLIT_STRATEGIES,
)

# Notebook display, cache-inspection, split-review, and demo helpers.
from scripts.session1_intro_utils import (
    choose_cache_bundle_paths,
    directory_size_bytes,
    derive_fractional_row_limit,
    evenly_spaced_take,
    filter_csv_paths_with_required_columns,
    load_raw_flag_context_sample,
    load_raw_scalar_sample,
    build_reviewed_modelling_split,
    show_reviewed_modelling_split_build,
    show_reviewed_model_row_accounting,
    show_fixed_split_review,
    show_cnn_interval_demo,
    show_episode_aware_split_comparison,
    summarize_issue_adequacy,
    build_split_strategy_tables,
    show_session1_cache_inspection,
    load_session1_cache_context,
    read_parquet_head,
    select_part_paths,
    stage_session1_inputs,
    show_reviewed_split_summary,
    show_session1_cache_read_comparison,
    show_split_strategy_comparison,
    show_split_strategy_timeline,
    show_temporal_flag_summary,
)

# Generic CSV/parquet cache helpers for participant-owned data.
from scripts.parquet_cache_utils import (
    csv_files_to_row_parquet_cache,
    resolve_csv_paths,
    resolve_or_create_parquet_cache,
)

# ONC-specific cache builders used by the more detailed data-preparation path.
from scripts.onc_scalar_cache_utils import (
    create_onc_row_level_parquet_cache,
    create_onc_window_summary_parquet_cache,
)
from scripts.prepare_scalar_session1_data import (
    DEFAULT_MEASUREMENT_COLUMNS,
    locate_header,
    read_scalar_csv,
)

## Start Here: Dataset And Cache Setup

Run the setup path that matches your data before any model section.

- **Option A** uses a pre-loaded workshop dataset and prepared parquet cache. This is the default path.
- **Option B** creates a parquet cache from your own CSV files. Run **Option C** afterwards to load that new cache.
- **Option C** loads an existing parquet cache, including one you just created with Option B.

The built-in dataset options are `ctd_conductivity`, `fluorometer_turbidity`, `oxygen`, and `conductivity_plugs`. Custom data needs a timestamp column, a target/label column, numeric measurement columns, and good/issue label definitions.

Workshop preset values live in `workshop_config/session1_dataset_profiles.py`. Edit that file if a prepared dataset needs a different target column, feature-column list, plot column, cache stem, or label grouping. Shared label meanings, colours, cache fallbacks, and default measurement-column lists live in `workshop_config/session1_defaults.py`.

If you want the step-by-step explanation of raw CSV inspection, parquet caching, reviewed-row accounting, or split design, work through `session1_data_preparation.ipynb` first. This notebook uses exactly the same setup as the machine-learning notebook, plus the optional episode-aware split.

### ✏️ Option A: Load A Prepared Workshop Dataset

Use this default path for the pre-loaded Session 1 datasets. The controls below do three things:

- choose the prepared dataset/cache,
- choose how much reviewed data to keep for an interactive run,
- choose the train/validation/test split and the optional train-only subsample.

Most runs only need `DATASET_PROFILE_ID`, `DATA_FRACTION`, `SPLIT_STRATEGY`, and `TRAIN_SUBSET_STRATEGY`. The remaining controls are visible so you can reproduce or override the dataset-building choices from the data-preparation notebook without digging through utility code.

This notebook adds one split option: `SPLIT_STRATEGY = "episode_aware"` keeps each issue episode and its surrounding clean rows in one split, with buffer rows removed at split boundaries. The optional episode-aware section in `session1_data_preparation.ipynb` compares it with the basic strategies.

In [ ]:
# ✏️ EDIT: Option A — built-in workshop dataset and prepared parquet cache.

# Dataset preset. Options: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
DATASET_PROFILE_ID = "ctd_conductivity"

# Optional path/cache overrides. Keep these as None for the workshop shared/local cache autodetection.
READ_RAW_DATA_DIR = None      # Example override: "/path/to/raw_csv_folder".
READ_CACHE_DIR = None         # Example override: "/path/to/prepared/parquet_cache".
CACHE_BUNDLE_NAME = None      # Example override: "my_prepared_cache".

# Dataset-size controls. DATA_FRACTION scales reviewed-row and train-subset caps.
# Use 0.1 for a quick run, 0.5 for a moderate run, 0.9 for a larger run, and 1.0 for no fraction cap.
DATA_FRACTION = 0.1
USE_DATA_FRACTION_BUDGETS = True  # False keeps every reviewed row and removes routine setup caps.

# Reviewed-row loading controls. None means "use the default derived from DATA_FRACTION" where applicable.
REVIEWED_FILE_SELECTION_MODE = "spread"  # Options: "spread", "first", "last", or "all".
REVIEWED_FILE_LIMIT = None                # Limit the number of parquet parts/files loaded, or None for all selected by the profile.
REVIEWED_MODEL_ROW_LIMIT = None           # Override the reviewed modelling row cap, or None to derive it from DATA_FRACTION.

# Train/validation/test split controls.
SPLIT_STRATEGY = "global_contiguous"      # Options: "global_contiguous", "per_source_contiguous", "interleaved_blocks", "episode_aware".
INTERLEAVED_BLOCK_ROWS = 1024             # Block size used only when SPLIT_STRATEGY = "interleaved_blocks".
TRAIN_FRACTION = 0.70                     # Fraction of reviewed modelling rows assigned to train.
VALIDATION_FRACTION = 0.15                # Fraction assigned to validation; the rest goes to test.

# Episode-aware split controls. Used only when SPLIT_STRATEGY = "episode_aware".
EPISODE_CONTEXT_ROWS = 512                # Clean rows kept in the same split as each issue episode.
EPISODE_MERGE_GAP_ROWS = 128              # Issue stretches closer than this merge into one episode.
EPISODE_PURGE_GAP_ROWS = 256              # Buffer rows removed at split boundaries to reduce leakage.

# Train-only subsampling controls. These affect train_df, not valid_df or test_df.
TRAIN_SUBSET_STRATEGY = "time_spread"     # Options: "time_spread", "issue_focused", "balanced_reviewed", "full_train".
TRAIN_SUBSET_MAX_ROWS = None              # Override max train rows, or None to derive it from DATA_FRACTION.
TRAIN_SUBSET_BLOCK_ROWS = 256             # Consecutive rows per kept block; at least the largest CNN/Transformer window_size.
ISSUE_ROWS_PER_FILE = None                # Used by "issue_focused"; None derives it from DATA_FRACTION.
BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25  # Used by "balanced_reviewed"; 0.25 means target about 25% issue rows.

# Reproducible sampling/training seed.
SEED = 21

# --- No edits needed below this line ---
# ML_STATE is the setup dictionary used by the rest of this notebook.
# The next setup cell exposes only the core modelling variables/dataframes.
ML_STATE = load_ml_section_state(
    dataset_profile_id=DATASET_PROFILE_ID,
    data_fraction=DATA_FRACTION,
    use_data_fraction_budgets=USE_DATA_FRACTION_BUDGETS,
    split_strategy=SPLIT_STRATEGY,
    train_subset_strategy=TRAIN_SUBSET_STRATEGY,
    reviewed_file_selection_mode=REVIEWED_FILE_SELECTION_MODE,
    reviewed_file_limit=REVIEWED_FILE_LIMIT,
    reviewed_model_row_limit=REVIEWED_MODEL_ROW_LIMIT,
    train_subset_max_rows=TRAIN_SUBSET_MAX_ROWS,
    train_subset_block_rows=TRAIN_SUBSET_BLOCK_ROWS,
    issue_rows_per_file=ISSUE_ROWS_PER_FILE,
    balanced_reviewed_target_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    interleaved_block_rows=INTERLEAVED_BLOCK_ROWS,
    episode_context_rows=EPISODE_CONTEXT_ROWS,
    episode_merge_gap_rows=EPISODE_MERGE_GAP_ROWS,
    episode_purge_gap_rows=EPISODE_PURGE_GAP_ROWS,
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    seed=SEED,
    read_raw_data_dir=READ_RAW_DATA_DIR,
    read_cache_dir=READ_CACHE_DIR,
    cache_bundle_name=CACHE_BUNDLE_NAME,
    verbose=True,
)

### ✏️ Option B: Create A Parquet Cache From Your Own CSV Files

Use this cell when your data is still in CSV files. The code is switched off by the `'''` lines at its top and bottom. To use it, delete those two lines, edit the paths/columns/labels, and run the cell to create a row-level parquet cache.

After this finishes, run **Option C** with the same cache folder/name and column settings to load the cache for modelling.

In [ ]:
# ✏️ EDIT: Option B — create a parquet cache from your own CSV files.
# This cell is switched off by the ''' lines. To use it, delete the ''' line below and the
# ''' line at the end of the cell, edit the settings, then run it.
'''
from pathlib import Path
import sys

# Find the repository root before importing helper functions from scripts/.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.parquet_cache_utils import resolve_or_create_parquet_cache

# CSV inputs. Use CUSTOM_RAW_CSV_DIR for every *.csv in a folder, or use
# CUSTOM_RAW_CSV_PATHS for an exact list of files. Leave the other one as None.
CUSTOM_RAW_CSV_DIR = Path("/path/to/csv_folder")
CUSTOM_RAW_CSV_PATHS = None  # Example: [Path("/path/to/file1.csv"), Path("/path/to/file2.csv")]

# Parquet output. CUSTOM_CACHE_BUNDLE_NAME becomes the filename stem, for example
# my_sensor_cache_metadata.json and my_sensor_cache_row_level/.
CUSTOM_PARQUET_CACHE_DIR = Path("/path/to/parquet_cache_folder")
CUSTOM_CACHE_BUNDLE_NAME = "my_sensor_cache"

# CSV-reading controls.
CUSTOM_TIME_COLUMN = "Time UTC"  # Timestamp column; change this if your CSV uses another name, such as "timestamp".
CUSTOM_CSV_HEADER = "first_row"  # Options: "first_row", "auto" for ONC-style metadata headers, or an integer row number such as 5.
CUSTOM_TARGET_FLAG = "label"  # Label column the supervised models should predict.
CUSTOM_MEASUREMENT_COLUMNS = ["sensor_value"]  # Numeric sensor columns used as ML features.
CUSTOM_OPTIONAL_QC_COLUMNS = []  # Extra context columns to preserve in parquet, or [] if none.
CUSTOM_ISSUE_LABELS = [1]  # Labels counted as issue/anomaly rows in cache summaries.

# Build the cache only if it is missing. Set force_rebuild_cache=True when you
# intentionally want to overwrite an existing cache with the same name.
cache_build_result = resolve_or_create_parquet_cache(
    cache_root=CUSTOM_PARQUET_CACHE_DIR,
    cache_bundle_name=CUSTOM_CACHE_BUNDLE_NAME,
    raw_data_dir=CUSTOM_RAW_CSV_DIR,
    raw_csv_paths=CUSTOM_RAW_CSV_PATHS,
    target_flag=CUSTOM_TARGET_FLAG,
    measurement_columns=CUSTOM_MEASUREMENT_COLUMNS,
    optional_qc_columns=CUSTOM_OPTIONAL_QC_COLUMNS,
    issue_labels=CUSTOM_ISSUE_LABELS,
    time_column=CUSTOM_TIME_COLUMN,
    csv_header=CUSTOM_CSV_HEADER,
    build_cache_if_missing=True,
    force_rebuild_cache=False,
    generic_csv_cache=True,
)
print(cache_build_result["summary"])

# Next step: run Option C below to load this cache into train/validation/test dataframes.
''';

### ✏️ Option C: Load An Existing Custom Parquet Cache

Use this cell when you already have a parquet cache. That cache might come from Option B above, the data-preparation notebook, or another run.

The code is switched off by the `'''` lines at its top and bottom. To use it, delete those two lines, edit the cache path/name and column settings, then run the cell before continuing to the ML sections.

In [ ]:
# ✏️ EDIT: Option C — load an existing parquet cache, then continue with the ML sections.
# This cell is switched off by the ''' lines. To use it, delete the ''' line below and the
# ''' line at the end of the cell, edit the settings, then run it.
'''
from pathlib import Path
import sys

# Find the repository root before importing helper functions from scripts/.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from scripts.session1_resume_utils import load_ml_section_state

# Cache location. These should match the values used when the cache was created.
CUSTOM_PARQUET_CACHE_DIR = Path("/path/to/cache_folder")
CUSTOM_CACHE_BUNDLE_NAME = "my_sensor_cache"

# Dataset-size controls used after the parquet cache is loaded.
DATA_FRACTION = 0.1  # 0.1 is quick, 0.5 is moderate, 0.9 is larger, and 1.0 removes row caps.
USE_DATA_FRACTION_BUDGETS = True  # False keeps every reviewed row and removes routine setup caps.
REVIEWED_FILE_SELECTION_MODE = "spread"  # Options: "spread", "first", "last", or "all".
REVIEWED_FILE_LIMIT = None  # Limit parquet parts/files, or None for all selected by the cache/profile.
REVIEWED_MODEL_ROW_LIMIT = None  # Override reviewed modelling row cap, or None to derive from DATA_FRACTION.

# Split controls.
SPLIT_STRATEGY = "global_contiguous"  # Options: "global_contiguous", "per_source_contiguous", "interleaved_blocks".
INTERLEAVED_BLOCK_ROWS = 1024  # Used only when SPLIT_STRATEGY = "interleaved_blocks".
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

# Train-only subsampling controls. These affect train_df, not valid_df or test_df.
TRAIN_SUBSET_STRATEGY = "time_spread"  # Options: "time_spread", "issue_focused", "balanced_reviewed", "full_train".
TRAIN_SUBSET_MAX_ROWS = None  # Override max train rows, or None to derive from DATA_FRACTION.
TRAIN_SUBSET_BLOCK_ROWS = 256  # Consecutive rows per kept block; at least the largest CNN/Transformer window_size.
ISSUE_ROWS_PER_FILE = None  # Used by "issue_focused"; None derives it from DATA_FRACTION.
BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25  # Used by "balanced_reviewed"; 0.25 means target about 25% issue rows.
SEED = 21  # Reproducible sampling/training seed.

# Column and label settings. These tell the ML sections what your parquet rows mean.
CUSTOM_TARGET_FLAG = "label"  # Label column the supervised models should predict.
CUSTOM_MEASUREMENT_COLUMNS = ["sensor_value"]  # Numeric sensor columns used by RF/CNN/Transformer/k-means.
CUSTOM_OPTIONAL_QC_COLUMNS = []  # Extra context columns to load, or [] if none.
CUSTOM_GOOD_LABELS = [0]  # Labels treated as normal/good examples.
CUSTOM_ISSUE_LABELS = [1]  # Labels treated as issue/anomaly examples.

# Plot columns. Usually choose one primary sensor column and, optionally, one context column.
CUSTOM_PLOT_MEASUREMENT_COLUMN = "sensor_value"
CUSTOM_PLOT_SECONDARY_COLUMN = None

# This creates ML_STATE from your custom cache. Run the next active cell afterwards
# so the Random Forest, k-means, CNN, and Transformer sections receive the same variables.
ML_STATE = load_ml_section_state(
    dataset_profile_id="custom",  # Custom template; explicit settings below define your dataset.
    data_fraction=DATA_FRACTION,
    use_data_fraction_budgets=USE_DATA_FRACTION_BUDGETS,
    split_strategy=SPLIT_STRATEGY,
    train_subset_strategy=TRAIN_SUBSET_STRATEGY,
    reviewed_file_selection_mode=REVIEWED_FILE_SELECTION_MODE,
    reviewed_file_limit=REVIEWED_FILE_LIMIT,
    reviewed_model_row_limit=REVIEWED_MODEL_ROW_LIMIT,
    train_subset_max_rows=TRAIN_SUBSET_MAX_ROWS,
    train_subset_block_rows=TRAIN_SUBSET_BLOCK_ROWS,
    issue_rows_per_file=ISSUE_ROWS_PER_FILE,
    balanced_reviewed_target_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    interleaved_block_rows=INTERLEAVED_BLOCK_ROWS,
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    seed=SEED,
    parquet_cache_dir=CUSTOM_PARQUET_CACHE_DIR,
    cache_bundle_name=CUSTOM_CACHE_BUNDLE_NAME,
    target_flag=CUSTOM_TARGET_FLAG,
    measurement_columns=CUSTOM_MEASUREMENT_COLUMNS,
    optional_qc_columns=CUSTOM_OPTIONAL_QC_COLUMNS,
    good_labels=CUSTOM_GOOD_LABELS,
    issue_labels=CUSTOM_ISSUE_LABELS,
    plot_measurement_column=CUSTOM_PLOT_MEASUREMENT_COLUMN,
    plot_secondary_column=CUSTOM_PLOT_SECONDARY_COLUMN,
    kmeans_feature_mode="row_level",  # Custom CSV caches usually do not include window-summary parquet.
    verbose=True,
)
''';

### Review The Loaded Dataset Choices

`ML_STATE` now holds the selected cache paths, label settings, reviewed modelling dataframe, train/validation/test split, and train-only subset. Later model sections read those values directly from `ML_STATE`, so there is one resolved setup dictionary instead of a second layer of notebook-wide copies.

Setup-time choices are applied when Option A or Option C creates `ML_STATE`. To change them, edit the controls in that option and rerun its load cell.

| Setting | What it controls | How to change it |
|---|---|---|
| `DATA_FRACTION` | Overall setup size knob for reviewed rows and routine train-subset caps | Edit `DATA_FRACTION` in Option A or Option C |
| `REVIEWED_FILE_SELECTION_MODE` / `REVIEWED_FILE_LIMIT` | Which parquet parts/files are loaded before reviewed rows are counted | Edit those controls; keep `limit=None` to use all selected parts |
| `REVIEWED_MODEL_ROW_LIMIT` | Maximum reviewed rows kept before split comparison | Leave `None` to derive from `DATA_FRACTION`, or set an integer row cap |
| `SPLIT_STRATEGY` | How reviewed rows become train/validation/test | Choose `"global_contiguous"`, `"per_source_contiguous"`, or `"interleaved_blocks"` |
| `INTERLEAVED_BLOCK_ROWS` | Block size for the interleaved split strategy | Edit when `SPLIT_STRATEGY = "interleaved_blocks"` |
| `EPISODE_CONTEXT_ROWS` / `EPISODE_MERGE_GAP_ROWS` / `EPISODE_PURGE_GAP_ROWS` | How the episode-aware split groups issue episodes and buffers split boundaries | Edit in Option A when `SPLIT_STRATEGY = "episode_aware"` |
| `TRAIN_SUBSET_MAX_ROWS` | Maximum training rows kept by the train-only subsampling step | Leave `None` to derive from `DATA_FRACTION`, or set an integer row cap |
| `TRAIN_SUBSET_BLOCK_ROWS` | Number of consecutive rows in each block kept by the train-only subset, so neighbouring training rows are as far apart in time as validation and test rows | Edit the integer; keep it at least as large as the CNN/Transformer `window_size`. Smaller blocks spread the subset over more separate stretches of time |
| `ISSUE_ROWS_PER_FILE` | Extra issue rows requested by the `issue_focused` train-subset strategy | Leave `None` to derive from `DATA_FRACTION`, or set an integer |
| `BALANCED_REVIEWED_TARGET_ISSUE_SHARE` | Target issue-row share for the `balanced_reviewed` train-subset strategy | Edit this fraction; `0.25` means target about 25% issue rows |

Experiment-specific row caps are defined in each experiment's settings cell, right before they are used.

## Part 1 — Random Forest: Tuning, Features, Tree Families, And Targets

The machine-learning notebook trained a baseline Random Forest. This part asks how to improve it:

1. tune the forest's hyperparameters on the validation split,
2. give the model more temporal context,
3. try other tree families,
4. and reconsider the target definition itself.

In many tabular problems, feature preparation and target definition matter more than squeezing a few extra points from `n_estimators` or `max_depth`.

### ✏️ Random Forest Row Budget For The Follow-Up Fits

After the search, the two best settings are retrained on a larger training sample and scored on validation and test. The next cell sets that training-row cap, using the same caps as the machine-learning notebook, and `RF_BLOCK_ROWS`, the length of the short consecutive-row blocks the forest's caps keep. The search itself has its own row caps in its settings cell.

In [ ]:
# ✏️ EDIT (optional): training-row cap for retraining the best search settings.
RF_TRAIN_BASE_ROWS = 250_000       # Training rows used when DATA_FRACTION = 1.0 and caps are on.
RF_TRAIN_MIN_ROWS = 25_000         # Smallest training cap after scaling by DATA_FRACTION.

if ML_STATE['USE_DATA_FRACTION_BUDGETS'] and ML_STATE['DATA_FRACTION'] < 0.999:
    RF_TRAIN_MAX_ROWS = max(RF_TRAIN_MIN_ROWS, int(RF_TRAIN_BASE_ROWS * ML_STATE['DATA_FRACTION']))
else:
    RF_TRAIN_MAX_ROWS = None

# The forest compares each row only with the row before it, so short blocks are enough.
# Shorter blocks spread the rows across more of the timeline; rare flags need that coverage.
# The search in the next cells uses the same block size.
RF_BLOCK_ROWS = 32

# Example override:
# RF_TRAIN_MAX_ROWS = None

# --- No edits needed below this line ---
# Reuse the active train-subset strategy when capping again. "full_train" becomes
# "time_spread" so a cap keeps rows across the whole training time span.
RF_TRAIN_CAP_STRATEGY = (
    ML_STATE['TRAIN_SUBSET_STRATEGY']
    if ML_STATE['TRAIN_SUBSET_STRATEGY'] != "full_train"
    else "time_spread"
)

show_setup_json(
    {
        "DATA_FRACTION": ML_STATE['DATA_FRACTION'],
        "RF_TRAIN_MAX_ROWS": RF_TRAIN_MAX_ROWS,
        "RF_TRAIN_CAP_STRATEGY": RF_TRAIN_CAP_STRATEGY,
        "RF_BLOCK_ROWS": RF_BLOCK_ROWS,
    }
)

### ✏️ Random Forest: Tune The Forest, Then Rethink The Features

The lesson in this part is deliberate:

1. start from the baseline forest you trained in the machine-learning notebook,
2. tune the forest hyperparameters on the validation split,
3. then ask the more important question: are we feeding the model the right information?

The next cell lists the settings to try. Each combination in `RF_SEARCH_SPACE` trains one forest, so the number of trials is the product of the list lengths.

In [ ]:
# ✏️ EDIT: Random Forest settings to try, and the row caps for the search.
RF_SEARCH_SPACE = {
    "n_estimators": [250, 400],
    "max_depth": [18, None],
    "min_samples_leaf": [1],
    "min_samples_split": [2],
    "max_features": ["sqrt"],
    "class_weight": ["balanced_subsample"],
}

# Search row caps live here because they only affect this RF tuning section.
# Set any *_MAX_ROWS value to None after this block if you want each search trial
# to use every row in that split.
RF_SEARCH_TRAIN_BASE_ROWS = 500_000
RF_SEARCH_TRAIN_MIN_ROWS = 200_000
RF_SEARCH_EVAL_BASE_ROWS = 150_000
RF_SEARCH_EVAL_MIN_ROWS = 15_000
if ML_STATE['USE_DATA_FRACTION_BUDGETS'] and ML_STATE['DATA_FRACTION'] < 0.999:
    RF_SEARCH_MODEL_ROW_LIMIT = max(RF_SEARCH_TRAIN_MIN_ROWS, int(RF_SEARCH_TRAIN_BASE_ROWS * ML_STATE['DATA_FRACTION']))
    RF_SEARCH_VALIDATION_MAX_ROWS = max(RF_SEARCH_EVAL_MIN_ROWS, int(RF_SEARCH_EVAL_BASE_ROWS * ML_STATE['DATA_FRACTION']))
    RF_SEARCH_TEST_MAX_ROWS = max(RF_SEARCH_EVAL_MIN_ROWS, int(RF_SEARCH_EVAL_BASE_ROWS * ML_STATE['DATA_FRACTION']))
else:
    RF_SEARCH_MODEL_ROW_LIMIT = None
    RF_SEARCH_VALIDATION_MAX_ROWS = None
    RF_SEARCH_TEST_MAX_ROWS = None

# --- No edits needed below this line ---
BEST_RF_SEARCH_CONFIG = None

print(
    {
        "DATA_FRACTION": ML_STATE['DATA_FRACTION'],
        "RF_SEARCH_MODEL_ROW_LIMIT": RF_SEARCH_MODEL_ROW_LIMIT,
        "RF_SEARCH_VALIDATION_MAX_ROWS": RF_SEARCH_VALIDATION_MAX_ROWS,
        "RF_SEARCH_TEST_MAX_ROWS": RF_SEARCH_TEST_MAX_ROWS,
    }
)

The search-space cell lists the Random Forest settings to try. The next cell runs those trials on the visible search row budget, compares validation results, and keeps the best configuration for the follow-up experiments.

In [ ]:
import gc

rf_search_train_selection_df = sample_frame_by_strategy(
    ML_STATE['train_full_df'],
    rows_limit=RF_SEARCH_MODEL_ROW_LIMIT,
    sample_strategy=ML_STATE['TRAIN_SUBSET_STRATEGY'] if ML_STATE['TRAIN_SUBSET_STRATEGY'] != "full_train" else "time_spread",
    target_flag=ML_STATE['TARGET_FLAG'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    issue_rows=ML_STATE['ISSUE_ROWS_PER_FILE'],
    balanced_issue_share=ML_STATE['BALANCED_REVIEWED_TARGET_ISSUE_SHARE'],
    block_rows=RF_BLOCK_ROWS,
)
rf_search_valid_selection_df = sample_frame_by_strategy(
    ML_STATE['valid_df'],
    rows_limit=RF_SEARCH_VALIDATION_MAX_ROWS,
    sample_strategy="time_spread",
    target_flag=ML_STATE['TARGET_FLAG'],
    block_rows=RF_BLOCK_ROWS,
)
rf_search_test_selection_df = sample_frame_by_strategy(
    ML_STATE['test_df'],
    rows_limit=RF_SEARCH_TEST_MAX_ROWS,
    sample_strategy="time_spread",
    target_flag=ML_STATE['TARGET_FLAG'],
    block_rows=RF_BLOCK_ROWS,
)
rf_search_labels = sorted(ML_STATE['reviewed_model_df']["model_target"].dropna().astype(int).unique().tolist())

rf_search_split_rows = materialize_reviewed_split_frames(
    ML_STATE['selected_paths'],
    {
        "train": rf_search_train_selection_df,
        "validation": rf_search_valid_selection_df,
        "test": rf_search_test_selection_df,
    },
    columns=ML_STATE['ROW_USE_COLUMNS'],
    target_flag=ML_STATE['TARGET_FLAG'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
)
rf_search_train_df, feature_columns, _ = build_model_frame(
    rf_search_split_rows["train"],
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
rf_search_valid_df, _, _ = build_model_frame(
    rf_search_split_rows["validation"],
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
globals().pop("rf_search_split_rows", None)
gc.collect()

rf_search_results, _, _ = run_rf_search(
    rf_search_train_df,
    rf_search_valid_df,
    feature_columns,
    labels=rf_search_labels,
    task_mode=ML_STATE['task_mode'],
    seed=ML_STATE['SEED'],
    search_space=RF_SEARCH_SPACE,
)
display(rf_search_results.head(10))

for _rf_name in ["rf_search_train_df", "rf_search_valid_df"]:
    globals().pop(_rf_name, None)
gc.collect()

rf_candidate_keys = [
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "min_samples_split",
    "max_features",
    "class_weight",
]
rf_transfer_train_selection_df = (
    sample_frame_by_strategy(
        ML_STATE['train_df'],
        rows_limit=RF_TRAIN_MAX_ROWS,
        sample_strategy=RF_TRAIN_CAP_STRATEGY,
        target_flag=ML_STATE['TARGET_FLAG'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        issue_rows=ML_STATE['ISSUE_ROWS_PER_FILE'],
        balanced_issue_share=ML_STATE['BALANCED_REVIEWED_TARGET_ISSUE_SHARE'],
        block_rows=RF_BLOCK_ROWS,
    )
    if RF_TRAIN_MAX_ROWS is not None and len(ML_STATE['train_df']) > RF_TRAIN_MAX_ROWS
    else ML_STATE['train_df'].copy()
)
rf_transfer_split_rows = materialize_reviewed_split_frames(
    ML_STATE['selected_paths'],
    {
        "train": rf_transfer_train_selection_df,
        "validation": rf_search_valid_selection_df,
        "test": rf_search_test_selection_df,
    },
    columns=ML_STATE['ROW_USE_COLUMNS'],
    target_flag=ML_STATE['TARGET_FLAG'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
)
rf_transfer_train_df, feature_columns, _ = build_model_frame(
    rf_transfer_split_rows["train"],
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
rf_transfer_valid_df, _, _ = build_model_frame(
    rf_transfer_split_rows["validation"],
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
rf_transfer_test_df, _, _ = build_model_frame(
    rf_transfer_split_rows["test"],
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
globals().pop("rf_transfer_split_rows", None)
gc.collect()

rf_transfer_rows = []
for _, candidate_row in rf_search_results.head(2).iterrows():
    candidate_config = {}
    for key in rf_candidate_keys:
        value = candidate_row[key]
        if key in {"n_estimators", "min_samples_leaf", "min_samples_split"}:
            candidate_config[key] = int(value)
        elif key == "max_depth":
            candidate_config[key] = None if pd.isna(value) else int(value)
        else:
            candidate_config[key] = value
    candidate_model = fit_random_forest(
        rf_transfer_train_df,
        feature_columns,
        seed=ML_STATE['SEED'],
        config=candidate_config,
    )
    candidate_valid = evaluate_classifier(
        candidate_model,
        rf_transfer_valid_df,
        feature_columns,
        labels=rf_search_labels,
        task_mode=ML_STATE['task_mode'],
    )
    candidate_test = evaluate_classifier(
        candidate_model,
        rf_transfer_test_df,
        feature_columns,
        labels=rf_search_labels,
        task_mode=ML_STATE['task_mode'],
    )
    rf_transfer_rows.append(
        {
            "config": candidate_config,
            "validation_f1": candidate_valid["f1"],
            "test_f1": candidate_test["f1"],
        }
    )
    del candidate_model
    gc.collect()

rf_transfer_frame = pd.DataFrame(rf_transfer_rows).sort_values("validation_f1", ascending=False).reset_index(drop=True)
BEST_RF_SEARCH_CONFIG = rf_transfer_frame.iloc[0]["config"]
display(rf_transfer_frame)

for _rf_name in [
    "rf_search_results",
    "rf_search_train_selection_df",
    "rf_search_valid_selection_df",
    "rf_search_test_selection_df",
    "rf_transfer_train_selection_df",
    "rf_transfer_train_df",
    "rf_transfer_valid_df",
    "rf_transfer_test_df",
    "rf_transfer_frame",
    "rf_transfer_rows",
]:
    globals().pop(_rf_name, None)
gc.collect()

print({"BEST_RF_SEARCH_CONFIG": BEST_RF_SEARCH_CONFIG})

### Beyond Random Forest: Better Features, Better Tree Families, Better Targets

The machine-learning notebook teaches the baseline Random Forest workflow. The next question is how to improve it without jumping straight to a neural network.

A useful way to think about tree-model improvement is:

1. **Add more context**. If a single row is not enough, give the model lags and rolling summaries.
2. **Try a stronger tree family**. Random Forest is not the only bagged-tree option; ExtraTrees can work better on noisy tabular data.
3. **Revisit the target itself**. If labels are ambiguous, collapsing some classes may produce a model that is both easier to learn and closer to the operational decision.

The next cell builds the temporal context features on a small training sample so you can see what they are. The cells after it summarise larger offline comparison runs when their result files are available in the artifact folder.

In [ ]:
# Build context features on a small, explicit training sample.
# This demonstrates the feature-engineering idea without carrying the full row cache in memory.
# Whole blocks of consecutive rows keep each row's real neighbours for the lag and rolling features.
context_selection_df = sample_frame_by_strategy(
    ML_STATE['train_df'],
    rows_limit=50_000,
    sample_strategy="time_spread",
    target_flag=ML_STATE['TARGET_FLAG'],
    block_rows=ML_STATE['TRAIN_SUBSET_BLOCK_ROWS'],
)
context_rows_df = load_selected_row_level_frame(
    ML_STATE['selected_paths'],
    context_selection_df,
    columns=ML_STATE['ROW_USE_COLUMNS'],
)
context_model_df, feature_columns, _ = build_model_frame(
    context_rows_df,
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
advanced_model_df, context_feature_columns = add_temporal_context_features(
    context_model_df,
    measurement_columns=ML_STATE['measurement_columns'],
)
lag_examples = [name for name in context_feature_columns if " lag_" in name][:12]
rolling_examples = [name for name in context_feature_columns if " roll_" in name][:12]
context_feature_examples = pd.DataFrame(
    [
        *[{"feature_group": "lag", "feature_name": name} for name in lag_examples],
        *[{"feature_group": "rolling", "feature_name": name} for name in rolling_examples],
    ]
)

print(
    {
        "context_sample_rows": len(advanced_model_df),
        "base_feature_count": len(feature_columns),
        "new_context_feature_count": len(context_feature_columns),
        "total_feature_count_if_used": len(feature_columns) + len(context_feature_columns),
    }
)
display(context_feature_examples)

for _context_name in ["context_selection_df", "context_rows_df", "context_model_df", "advanced_model_df"]:
    globals().pop(_context_name, None)
gc.collect()

### Tree Family Comparison

We compared:

- the baseline Random Forest,
- Random Forest plus temporal context features,
- ExtraTrees plus the same context features,
- and a gradient-boosted tree baseline.

The key point is not just *which model won*. It is *why*:

- context features gave the tree models access to short-term history,
- ExtraTrees handled the noisy tabular signal better than the baseline forest,
- and we checked performance on the same time-based splits rather than trusting one lucky number.

In [ ]:
tree_search_path = ML_STATE["ARTIFACT_DIR"] / "tree_model_comparison_search.csv"
tree_full_path = ML_STATE["ARTIFACT_DIR"] / "tree_model_comparison_full.csv"

if tree_search_path.exists() and tree_full_path.exists():
    tree_search_results = pd.read_csv(tree_search_path)
    tree_full_results = pd.read_csv(tree_full_path)

    display(
        tree_search_results[
            [
                "name",
                "family",
                "feature_variant",
                "validation_f1",
                "test_f1",
                "validation_class_4_recall",
                "test_class_4_recall",
            ]
        ]
    )
    display(
        tree_full_results[
            [
                "name",
                "family",
                "feature_variant",
                "validation_f1",
                "test_f1",
                "validation_class_4_recall",
                "test_class_4_recall",
            ]
        ]
    )
else:
    print(
        "Advanced tree comparison artifacts are not included in this repo. "
        "This section uses offline comparison results when they are available."
    )

### Target Design: Should We Collapse Ambiguous Flags?

Another important modelling decision is whether the original label set is the right one.

For this dataset, `3` and `4` are both problem cases, while `9` means the value is missing. Depending on the operational question, it can be more useful to predict:

- the original `1 / 3 / 4 / 9` labels,
- a collapsed `1 / 34 / 9` target,
- or a binary `good vs issue` target.

This is not just a modelling trick. It is a problem-definition question. A simpler target can be much easier to learn and may align better with the real decision you need to automate.

In [ ]:
strategy_summary_path = ML_STATE["ARTIFACT_DIR"] / "extratrees_strategy_summary.json"
strategy_results_path = ML_STATE["ARTIFACT_DIR"] / "extratrees_target_strategy_comparison.csv"
collapsed_metrics_path = ML_STATE["ARTIFACT_DIR"] / "collapsed_target_per_class_metrics.md"

if strategy_results_path.exists():
    strategy_results = pd.read_csv(strategy_results_path)
    display(strategy_results)
else:
    print(
        "Target-strategy comparison artifacts are not included in this repo. "
        "This section uses offline comparison results when they are available."
    )

per_class_tables = []
for csv_name, strategy_name, split_name in [
    ("collapsed_1_34_9_validation_per_class_metrics.csv", "collapsed_1_34_9", "validation"),
    ("collapsed_1_34_9_test_per_class_metrics.csv", "collapsed_1_34_9", "test"),
    ("binary_issue_validation_per_class_metrics.csv", "binary_issue", "validation"),
    ("binary_issue_test_per_class_metrics.csv", "binary_issue", "test"),
]:
    csv_path = ML_STATE["ARTIFACT_DIR"] / csv_name
    if csv_path.exists():
        frame = pd.read_csv(csv_path)
        frame.insert(0, "strategy", strategy_name)
        frame.insert(1, "split", split_name)
        per_class_tables.append(frame)

if per_class_tables:
    display(pd.concat(per_class_tables, ignore_index=True))
else:
    print(
        "Per-class collapsed-target metric artifacts are not included in this repo."
    )

### What We Learned From The Advanced Tree Experiments

The main lessons from these local experiments were:

- **More data helped**, but only modestly for the full 4-class problem.
- **Temporal context features helped more** because the QC issues often occur in short coherent runs.
- **ExtraTrees outperformed the baseline Random Forest** in this setting.
- **Collapsing ambiguous labels helped dramatically**, especially for `1 / 34 / 9` and binary `issue` detection.

This is exactly the kind of modelling mindset we want in Session 2:

- do not assume the first feature set is enough,
- do not assume the first model family is the best,
- and do not assume the original label definition is the most useful one.

## Part 2 — CNN: Search And Per-Timestamp Sequence Labelling

The machine-learning notebook trained one small baseline CNN on raw measurements. This part asks two follow-up questions:

- does a hyperparameter search improve the baseline window-classification CNN?
- does a deeper CNN that predicts one QC flag per timestamp, with extra change features, do better than tuning alone?

### Load PyTorch

The Random Forest part does not need PyTorch, so we import it here. The next cell also reports whether a GPU is available.

In [ ]:
# Run this cell when you are ready to work through the CNN section.
# Earlier notebook parts do not need PyTorch, so we import it here.
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    PYTORCH_READY = True
    CNN_READY = True
    print({
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })
except ImportError as exc:
    PYTORCH_READY = False
    CNN_READY = False
    print("PyTorch is not installed in this environment.")
    raise SystemExit(exc)

### ✏️ CNN: Tune The Network, Then Revisit The Problem Definition

We start by tuning the baseline window-classification CNN from the machine-learning notebook on the same target. That gives us a fair neural-network analogue to the Random Forest search.

Then we step back and ask a broader question:

- if feature engineering and target design mattered so much for trees, do they matter for neural networks too?

The answer turns out to be yes. The later CNN cells show that architecture knobs alone were not the whole story.

In [ ]:
# ✏️ EDIT: CNN settings to try, and the row caps for the search.
# This small grid search tries a few CNN sizes and learning rates.
# Keep it compact because each trial trains a neural network.
CNN_SEARCH_SPACE = {
    "window_size": [128, 256],
    "label_reduction": ["worst"],
    "epochs": [12],
    "batch_size": [128],
    "learning_rate": [5e-4, 1e-3],
    "weight_decay": [1e-4],
    "patience": [4],
    "min_delta": [0.001],
    "dropout": [0.15],
    "conv_channels": [[32, 64], [64, 128]],
    "lr_decay_factor": [0.5],
    "lr_patience": [1],
    "gradient_clip_norm": [1.0],
    "num_workers": [0],
    "pin_memory": [True],
    "persistent_workers": [False],
    "prefetch_factor": [2],
}

# Search row caps live here because they only affect this CNN tuning section.
# Set any *_MAX_ROWS value to None after this block if you want each search trial
# to use every row in that split.
CNN_SEARCH_TRAIN_BASE_ROWS = 500_000
CNN_SEARCH_TRAIN_MIN_ROWS = 200_000
CNN_SEARCH_EVAL_BASE_ROWS = 150_000
CNN_SEARCH_EVAL_MIN_ROWS = 15_000
if ML_STATE['USE_DATA_FRACTION_BUDGETS'] and ML_STATE['DATA_FRACTION'] < 0.999:
    CNN_SEARCH_MODEL_ROW_LIMIT = max(CNN_SEARCH_TRAIN_MIN_ROWS, int(CNN_SEARCH_TRAIN_BASE_ROWS * ML_STATE['DATA_FRACTION']))
    CNN_SEARCH_VALIDATION_MAX_ROWS = max(CNN_SEARCH_EVAL_MIN_ROWS, int(CNN_SEARCH_EVAL_BASE_ROWS * ML_STATE['DATA_FRACTION']))
    CNN_SEARCH_TEST_MAX_ROWS = max(CNN_SEARCH_EVAL_MIN_ROWS, int(CNN_SEARCH_EVAL_BASE_ROWS * ML_STATE['DATA_FRACTION']))
else:
    CNN_SEARCH_MODEL_ROW_LIMIT = None
    CNN_SEARCH_VALIDATION_MAX_ROWS = None
    CNN_SEARCH_TEST_MAX_ROWS = None

# Each cap keeps blocks of this many consecutive rows. Keep it at least twice the largest
# window_size above, so one dropped row (a missing reading) costs one window, not a whole block.
CNN_SEARCH_BLOCK_ROWS = 512

# --- No edits needed below this line ---
BEST_CNN_SEARCH_CONFIG = None

print(
    {
        "DATA_FRACTION": ML_STATE['DATA_FRACTION'],
        "CNN_SEARCH_MODEL_ROW_LIMIT": CNN_SEARCH_MODEL_ROW_LIMIT,
        "CNN_SEARCH_VALIDATION_MAX_ROWS": CNN_SEARCH_VALIDATION_MAX_ROWS,
        "CNN_SEARCH_TEST_MAX_ROWS": CNN_SEARCH_TEST_MAX_ROWS,
        "CNN_SEARCH_BLOCK_ROWS": CNN_SEARCH_BLOCK_ROWS,
    }
)

In [ ]:
import gc

# Search on a representative training subset, while keeping capped validation/test slices fixed for comparison.
# Every split keeps whole blocks of consecutive rows, so CNN windows span the same time in train, validation, and test.
cnn_search_train_selection_df = sample_frame_by_strategy(
    ML_STATE['train_full_df'],
    rows_limit=CNN_SEARCH_MODEL_ROW_LIMIT,
    sample_strategy=ML_STATE['TRAIN_SUBSET_STRATEGY'] if ML_STATE['TRAIN_SUBSET_STRATEGY'] != "full_train" else "time_spread",
    target_flag=ML_STATE['TARGET_FLAG'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    issue_rows=ML_STATE['ISSUE_ROWS_PER_FILE'],
    balanced_issue_share=ML_STATE['BALANCED_REVIEWED_TARGET_ISSUE_SHARE'],
    block_rows=CNN_SEARCH_BLOCK_ROWS,
)
cnn_search_valid_selection_df = sample_frame_by_strategy(
    ML_STATE['valid_df'],
    rows_limit=CNN_SEARCH_VALIDATION_MAX_ROWS,
    sample_strategy="time_spread",
    target_flag=ML_STATE['TARGET_FLAG'],
    block_rows=CNN_SEARCH_BLOCK_ROWS,
)
cnn_search_test_selection_df = sample_frame_by_strategy(
    ML_STATE['test_df'],
    rows_limit=CNN_SEARCH_TEST_MAX_ROWS,
    sample_strategy="time_spread",
    target_flag=ML_STATE['TARGET_FLAG'],
    block_rows=CNN_SEARCH_BLOCK_ROWS,
)
cnn_search_split_frames = materialize_reviewed_split_frames(
    ML_STATE['selected_paths'],
    {
        "train": cnn_search_train_selection_df,
        "validation": cnn_search_valid_selection_df,
        "test": cnn_search_test_selection_df,
    },
    columns=ML_STATE['ROW_USE_COLUMNS'],
    target_flag=ML_STATE['TARGET_FLAG'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
)
# The utility trains each candidate config and returns the best validation result.
cnn_search_results, best_cnn_config, best_cnn_result = run_cnn_search_from_frames(
    cnn_search_split_frames,
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    seed=ML_STATE['SEED'],
    search_space=CNN_SEARCH_SPACE,
    checkpoint_dir=ML_STATE["ARTIFACT_DIR"] / "notebook_cnn_search",
)
display(cnn_search_results.head(10))
# Store the winner so later cells can reuse or compare against it.
BEST_CNN_SEARCH_CONFIG = best_cnn_config
display(
    pd.DataFrame(
        [
            {
                "model": "Baseline 1D CNN search winner",
                "best_validation_f1": best_cnn_result["best_validation_f1"],
                "search_rows": len(cnn_search_split_frames["train"]),
                "config": BEST_CNN_SEARCH_CONFIG,
            }
        ]
    )
)

for _cnn_name in [
    "cnn_search_train_selection_df",
    "cnn_search_valid_selection_df",
    "cnn_search_test_selection_df",
    "cnn_search_split_frames",
    "cnn_search_results",
]:
    globals().pop(_cnn_name, None)
gc.collect()

print({"BEST_CNN_SEARCH_CONFIG": BEST_CNN_SEARCH_CONFIG})

### CNN: Does The Same Lesson Hold?

We can ask the same question for neural networks that we asked for Random Forest:

- does hyperparameter tuning help?
- or do target definition and input representation matter even more?

The answer from the larger local GPU study was: **the same lesson still holds**.

CNN hyperparameters matter, but the strongest gains came from:

1. choosing a target that matched the operational problem better,
2. giving the network more informative inputs,
3. and then tuning the neural-network settings on top of that.

In [ ]:
# Load saved CNN study outputs if they exist, so the notebook can summarise larger offline runs.
cnn_result_frames = []
for csv_name, study_name in [
    ("cnn_optimization_final.csv", "final_screen"),
    ("cnn_optimization_finalists.csv", "large_retrain"),
    ("cnn_targeted_large_results.csv", "targeted_large_retrain"),
]:
    csv_path = ML_STATE["ARTIFACT_DIR"] / csv_name
    if csv_path.exists():
        frame = pd.read_csv(csv_path)
        frame["study_name"] = study_name
        cnn_result_frames.append(frame)

if cnn_result_frames:
    # Combine all available study CSVs and rank by held-out test F1.
    cnn_results = pd.concat(cnn_result_frames, ignore_index=True, sort=False)
    cnn_results = cnn_results.sort_values(["test_f1", "validation_f1"], ascending=False).reset_index(drop=True)
    display(
        cnn_results[
            [
                "study_name",
                "strategy",
                "feature_set",
                "model_mode",
                "row_count",
                "window_size",
                "conv_channels",
                "validation_f1",
                "test_f1",
            ]
        ].head(12)
    )

    # These recommendations summarise which CNN variants were most reliable in the offline studies.
    robust_rows = [
        {
            "recommended_choice": "Best robust collapsed-target CNN",
            "target_strategy": "collapsed_1_34_9",
            "feature_set": "measurements_plus_deltas",
            "model_mode": "sequence",
            "window_size": 256,
            "why": "Best larger-sample CNN result; deltas capture sudden local changes, and 256 steps gives more temporal context.",
        },
        {
            "recommended_choice": "Best robust binary CNN",
            "target_strategy": "binary_issue",
            "feature_set": "measurements_plus_deltas_plus_rolling_time",
            "model_mode": "sequence",
            "window_size": 128,
            "why": "Strong for good-versus-issue decisions, but still below the tree baseline.",
        },
    ]
    display(pd.DataFrame(robust_rows))
else:
    print("CNN optimisation artifacts not found. Run the local CNN study scripts first.")

# Compare the best CNN studies with the tree baseline when that artifact is available.
comparison_path = ML_STATE["ARTIFACT_DIR"] / "extratrees_target_strategy_comparison.csv"
if cnn_result_frames and comparison_path.exists():
    tree_results = pd.read_csv(comparison_path)
    comparison_rows = [
        {
            "target_strategy": "collapsed_1_34_9",
            "best_cnn_test_f1": 0.9932348935330678,
            "best_tree_test_f1": float(
                tree_results.loc[tree_results["strategy"] == "collapsed_1_34_9", "test_f1"].iloc[0]
            ),
        },
        {
            "target_strategy": "binary_issue",
            "best_cnn_test_f1": 0.9855123210924848,
            "best_tree_test_f1": float(
                tree_results.loc[tree_results["strategy"] == "binary_issue", "test_f1"].iloc[0]
            ),
        },
        {
            "target_strategy": "multiclass_1_3_4_9",
            "best_cnn_test_f1": 0.6952206110852922,
            "best_tree_test_f1": float(
                tree_results.loc[tree_results["strategy"] == "multiclass_1_3_4_9", "test_f1"].iloc[0]
            ),
        },
    ]
    comparison_frame = pd.DataFrame(comparison_rows)
    comparison_frame["cnn_minus_tree"] = comparison_frame["best_cnn_test_f1"] - comparison_frame["best_tree_test_f1"]
    display(comparison_frame)

#### Why The Advanced CNN Defaults Below Use The Collapsed Target

The defaults in the next section are based on the most robust local GPU result we found, not just the highest one-off validation score.

Those defaults make three deliberate choices:

- **Target strategy:** dataset-aware by default
  For conductivity and turbidity, the advanced section starts from `collapsed_1_34_9`. For oxygen, it starts from `collapsed_12_34_9` so that `1/2` and `3/4` can travel together instead of reusing the CTD collapse blindly. For conductivity plugs, it starts from `raw_multiclass` because the `ml_label` classes follow a different scheme.
- **Input set:** `measurements_plus_deltas`
  Raw values tell us where the instrument is. Absolute deltas tell us when it changes abruptly. That combination was more reliable than raw values alone.
- **Model mode:** `sequence`
  QC is naturally a timestamp-level task, so per-timestamp prediction is a better conceptual match than forcing one label onto a whole window.

These are not magic settings. They are simply the choices that held up best when we reran the strongest candidates on larger samples.

### Advanced CNN Variant: Predict One QC Flag Per Timestamp

The baseline CNN in the machine-learning notebook is deliberately small and sees only raw measurements. This variant changes three things at once:

- **output:** one QC prediction for every time step inside the window (sequence labelling),
- **inputs:** raw measurements plus `abs_delta` change features, so sudden jumps are explicit,
- **capacity:** a deeper stack of convolution layers with batch normalisation.

Per-timestamp prediction is often a better match for the real ONC QC task, because the operational question is usually closer to "what is the flag at this timestamp?" than "what is the single label for this whole chunk?"

#### ✏️ Sequence-Labelling CNN Settings

This model uses almost the same ingredients as the baseline CNN, but the final layer produces a prediction at **every position in the window**.

Important differences from the earlier CNN:

- `sequence_target_strategy` lets us change the problem definition itself.
- `sequence_feature_set` lets us decide what information the CNN receives.
- `window_size` still controls context, but now every row in the window gets its own prediction.
- evaluation is done per timestamp, not per window.
- `loss` and `focal_gamma` choose the training loss, exactly as in the machine-learning notebook. Part 3 compares all four options on this model.
- `patience` and `min_delta` control early stopping: training stops when validation F1 has not improved by at least `min_delta` for `patience` epochs in a row.
- `indicator_channels` adds the dataset preset's 0/1 range-check channels (for turbidity, "turbidity below 0"), as explained in CNN Step 1 of the machine-learning notebook.

The defaults here mirror the strongest **robust** CNN run we found for the current problem framing. For conductivity and turbidity that means collapsed `1 / 34 / 9`; for oxygen it starts from `12 / 34 / 9`; for conductivity plugs it stays in raw multiclass mode until you decide whether a custom collapse is more helpful.

In [ ]:
# ✏️ EDIT: sequence-labelling CNN settings.
# The sequence-labelling CNN predicts a QC label at each timestamp instead of one label per window.
SEQUENCE_CNN_TARGET_STRATEGY = ML_STATE['DEFAULT_SEQUENCE_TARGET_STRATEGY']
SEQUENCE_CNN_FEATURE_SET = "measurements_plus_deltas"

# These settings use a slightly larger CNN because per-timestep prediction is a harder task.
SEQUENCE_CNN_CONFIG = {
    "window_size": 256,
    "epochs": 15,
    "batch_size": 192,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.0,
    "dropout": 0.15,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
    "conv_channels": [64, 128, 256],
    # Extra 0/1 input channels for range checks, from the dataset preset. Set to [] to leave them out.
    "indicator_channels": ML_STATE['SEQUENCE_INDICATOR_CHANNELS'],
    # Loss options: "weighted_cross_entropy", "sqrt_weighted_cross_entropy", "cross_entropy", or "focal".
    "loss": "weighted_cross_entropy",
    "focal_gamma": 2.0,  # Only used when loss="focal".
}

# Loader settings mirror the baseline CNN settings for easier comparison.
SEQUENCE_CNN_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": True,
    "persistent_workers": False,
    "prefetch_factor": 2,
}

SEQUENCE_CNN_MODEL_PATH = ML_STATE["ARTIFACT_DIR"] / "best_sequence_label_cnn_checkpoint.pt"

# Per-timestamp windows are much heavier than one-label-per-window CNN data.
# These caps live here because they only affect this sequence-CNN section.
# Set any *_MAX_ROWS value to None after this block if you want this section to
# use every row in that split.
SEQUENCE_CNN_TRAIN_BASE_ROWS = 1_000_000
SEQUENCE_CNN_TRAIN_MIN_ROWS = 10_000
SEQUENCE_CNN_EVAL_BASE_ROWS = 150_000
SEQUENCE_CNN_EVAL_MIN_ROWS = 15_000
SEQUENCE_CNN_TRAIN_CAP_CEILING = 80_000
SEQUENCE_CNN_EVAL_CAP_CEILING = 40_000
if ML_STATE['USE_DATA_FRACTION_BUDGETS'] and ML_STATE['DATA_FRACTION'] < 0.999:
    sequence_train_scaled_rows = max(SEQUENCE_CNN_TRAIN_MIN_ROWS, int(SEQUENCE_CNN_TRAIN_BASE_ROWS * ML_STATE['DATA_FRACTION']))
    sequence_eval_scaled_rows = max(SEQUENCE_CNN_EVAL_MIN_ROWS, int(SEQUENCE_CNN_EVAL_BASE_ROWS * ML_STATE['DATA_FRACTION']))
else:
    sequence_train_scaled_rows = SEQUENCE_CNN_TRAIN_CAP_CEILING
    sequence_eval_scaled_rows = SEQUENCE_CNN_EVAL_CAP_CEILING

SEQUENCE_CNN_TRAIN_MAX_ROWS = min(sequence_train_scaled_rows, SEQUENCE_CNN_TRAIN_CAP_CEILING)
SEQUENCE_CNN_VALIDATION_MAX_ROWS = min(sequence_eval_scaled_rows, SEQUENCE_CNN_EVAL_CAP_CEILING)
SEQUENCE_CNN_TEST_MAX_ROWS = min(sequence_eval_scaled_rows, SEQUENCE_CNN_EVAL_CAP_CEILING)

# Each cap keeps blocks of this many consecutive rows. Keep it at least twice window_size,
# so one dropped row (a missing reading) costs one window, not a whole block.
SEQUENCE_CNN_BLOCK_ROWS = 512

# --- No edits needed below this line ---
print(
    {
        "sequence_target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
        "sequence_feature_set": SEQUENCE_CNN_FEATURE_SET,
        "sequence_train_max_rows": SEQUENCE_CNN_TRAIN_MAX_ROWS,
        "sequence_validation_max_rows": SEQUENCE_CNN_VALIDATION_MAX_ROWS,
        "sequence_test_max_rows": SEQUENCE_CNN_TEST_MAX_ROWS,
        "sequence_train_cap_ceiling": SEQUENCE_CNN_TRAIN_CAP_CEILING,
        "sequence_eval_cap_ceiling": SEQUENCE_CNN_EVAL_CAP_CEILING,
    }
)
display(pd.Series(SEQUENCE_CNN_CONFIG, name="value").rename_axis("sequence_cnn_parameter").to_frame())
display(pd.Series(SEQUENCE_CNN_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())

### Step 1: Build The Target And Feature Set We Actually Want To Learn

For the baseline CNN, we reduced one whole window to a single label. Here we keep the label sequence itself, so a `256`-step input window has `256` targets.

We also make two explicit choices that came from the larger optimisation study:

- apply a target strategy before training,
- and decide which engineered inputs we want the CNN to see.

In [ ]:
import gc

# Release large arrays left by earlier cells before building per-timestep windows.
# This keeps 16 GB Narval/FIR jobs stable.
for _cnn_name in [
    "cnn_bundle",
    "X_train", "X_valid", "X_test", "y_train", "y_valid", "y_test",
    "train_targets_tensor", "valid_targets_tensor", "test_targets_tensor",
    "train_dataset", "valid_dataset", "test_dataset",
    "train_loader", "valid_loader", "test_loader",
    "cnn_model", "optimizer", "scheduler", "best_state", "loss_fn",
    "sequence_split_frames_materialized",
    "channel_mean", "channel_std",
]:
    globals().pop(_cnn_name, None)
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

if not CNN_READY:
    SEQUENCE_CNN_RUN = False
    print("Sequence-labelling CNN skipped because PyTorch is not available.")
else:
    # Per-timestep labels create many more target values than the baseline CNN.
    # Build this section from bounded split selections instead of reusing the
    # full row-level frames from the baseline CNN.
    sequence_train_selection_df = sample_frame_by_strategy(
        ML_STATE['train_full_df'],
        rows_limit=SEQUENCE_CNN_TRAIN_MAX_ROWS,
        sample_strategy=ML_STATE['TRAIN_SUBSET_STRATEGY'] if ML_STATE['TRAIN_SUBSET_STRATEGY'] != "full_train" else "time_spread",
        target_flag=ML_STATE['TARGET_FLAG'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        issue_rows=ML_STATE['ISSUE_ROWS_PER_FILE'],
        balanced_issue_share=ML_STATE['BALANCED_REVIEWED_TARGET_ISSUE_SHARE'],
        block_rows=SEQUENCE_CNN_BLOCK_ROWS,
    )
    sequence_validation_selection_df = sample_frame_by_strategy(
        ML_STATE['valid_df'],
        rows_limit=SEQUENCE_CNN_VALIDATION_MAX_ROWS,
        sample_strategy="time_spread",
        target_flag=ML_STATE['TARGET_FLAG'],
        block_rows=SEQUENCE_CNN_BLOCK_ROWS,
    )
    sequence_test_selection_df = sample_frame_by_strategy(
        ML_STATE['test_df'],
        rows_limit=SEQUENCE_CNN_TEST_MAX_ROWS,
        sample_strategy="time_spread",
        target_flag=ML_STATE['TARGET_FLAG'],
        block_rows=SEQUENCE_CNN_BLOCK_ROWS,
    )
    sequence_source_frames = materialize_reviewed_split_frames(
        ML_STATE['selected_paths'],
        {
            "train": sequence_train_selection_df,
            "validation": sequence_validation_selection_df,
            "test": sequence_test_selection_df,
        },
        columns=ML_STATE['ROW_USE_COLUMNS'],
        target_flag=ML_STATE['TARGET_FLAG'],
        task_mode=ML_STATE['task_mode'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
    )

    def build_sequence_feature_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
        feature_df = frame.sort_values("Time UTC").reset_index(drop=True).copy()
        # Absolute deltas help the CNN notice sudden changes between adjacent readings.
        # The first row after a time gap (such as the jump between two kept row blocks) gets 0.
        segment_ids = time_gap_segment_ids(feature_df)
        for column in ML_STATE['measurement_columns']:
            feature_df[f"{column} abs_delta"] = feature_df[column].groupby(segment_ids).diff().abs().fillna(0.0)

        # Rolling context features give each timestamp a small amount of local history.
        context_source = feature_df[["Time UTC", *ML_STATE['measurement_columns']]].copy()
        context_frame, context_columns = add_temporal_context_features(
            context_source,
            measurement_columns=ML_STATE['measurement_columns'],
            lag_steps=(),
            rolling_windows=(5,),
        )
        for column in context_columns:
            feature_df[column] = context_frame[column]

        # Cyclical time features avoid artificial jumps at midnight or year boundaries.
        radians_hour = 2.0 * np.pi * feature_df["Time UTC"].dt.hour / 24.0
        radians_doy = 2.0 * np.pi * feature_df["Time UTC"].dt.dayofyear / 366.0
        feature_df["hour_sin"] = np.sin(radians_hour)
        feature_df["hour_cos"] = np.cos(radians_hour)
        feature_df["doy_sin"] = np.sin(radians_doy)
        feature_df["doy_cos"] = np.cos(radians_doy)

        target_df, _, _ = apply_target_strategy(
            feature_df.dropna(subset=[ML_STATE['TARGET_FLAG']]).copy(),
            ML_STATE['TARGET_FLAG'],
            SEQUENCE_CNN_TARGET_STRATEGY,
            issue_labels=ML_STATE['ISSUE_LABELS'],
        )
        if "model_target" in target_df.columns:
            target_df = target_df.drop(columns=["model_target"])
        target_df = target_df.rename(columns={"strategy_target": "model_target"})
        return target_df, context_columns

    sequence_split_frames_full = {}
    sequence_context_columns = []
    for split_name, source_frame in sequence_source_frames.items():
        sequence_split_frames_full[split_name], split_context_columns = build_sequence_feature_frame(source_frame)
        if not sequence_context_columns:
            sequence_context_columns = split_context_columns

    # Optional 0/1 range-check channels from the dataset preset (for turbidity: turbidity below 0).
    for split_name in sequence_split_frames_full:
        sequence_split_frames_full[split_name], sequence_indicator_columns = add_indicator_channels(
            sequence_split_frames_full[split_name],
            SEQUENCE_CNN_CONFIG["indicator_channels"],
        )

    # Feature sets let us compare raw measurements against richer temporal representations.
    sequence_feature_sets = {
        "measurements": ML_STATE['measurement_columns'],
        "measurements_plus_deltas": ML_STATE['measurement_columns'] + [f"{column} abs_delta" for column in ML_STATE['measurement_columns']],
        "measurements_plus_deltas_plus_rolling_time": ML_STATE['measurement_columns']
        + [f"{column} abs_delta" for column in ML_STATE['measurement_columns']]
        + sequence_context_columns
        + ["hour_sin", "hour_cos", "doy_sin", "doy_cos"],
    }

    if SEQUENCE_CNN_FEATURE_SET not in sequence_feature_sets:
        raise ValueError(
            f"Unsupported sequence feature set: {SEQUENCE_CNN_FEATURE_SET}. Choose from {list(sequence_feature_sets)}."
        )

    _, sequence_class_labels, sequence_average = apply_target_strategy(
        sequence_source_frames["train"].dropna(subset=[ML_STATE['TARGET_FLAG']]).copy(),
        ML_STATE['TARGET_FLAG'],
        SEQUENCE_CNN_TARGET_STRATEGY,
        issue_labels=ML_STATE['ISSUE_LABELS'],
    )
    sequence_input_columns = sequence_feature_sets[SEQUENCE_CNN_FEATURE_SET] + sequence_indicator_columns
    sequence_split_frames = {
        "train": sequence_split_frames_full["train"],
        "validation": sequence_split_frames_full["validation"],
        "test": sequence_split_frames_full["test"],
    }

    sequence_window_size = SEQUENCE_CNN_CONFIG["window_size"]
    sequence_task_mode = "binary" if sequence_average == "binary" else "multiclass"
    # Build per-timestep windows: X is a sequence of features, y is a sequence of labels.
    sequence_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames,
        measurement_columns=sequence_input_columns,
        target_column="model_target",
        task_mode=sequence_task_mode,
        output_mode="per_timestep",
        window_size=sequence_window_size,
        label_reduction="worst",
    )

    sequence_class_labels = sequence_bundle.class_labels
    # As with the baseline CNN, each split needs complete windows and enough labels to learn.
    if sequence_task_mode == "multiclass" and len(sequence_class_labels) < 2:
        SEQUENCE_CNN_RUN = False
        print(
            {
                "split_strategy": ML_STATE['FIXED_SPLIT_STRATEGY'],
                "skip_reason": "Need at least two active classes to train the sequence-labelling CNN in multiclass mode.",
                "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                "feature_set": SEQUENCE_CNN_FEATURE_SET,
                "active_labels": sequence_class_labels,
            }
        )
    elif len(sequence_bundle.X_train) == 0 or len(sequence_bundle.X_valid) == 0 or len(sequence_bundle.X_test) == 0:
        SEQUENCE_CNN_RUN = False
        print(
            {
                "split_strategy": ML_STATE['FIXED_SPLIT_STRATEGY'],
                "skip_reason": "At least one split produced no full windows for this configuration.",
                "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                "feature_set": SEQUENCE_CNN_FEATURE_SET,
            }
        )
    else:
        # Conv1D expects channel-first arrays: (windows, channels, time).
        X_train_seq = np.transpose(sequence_bundle.X_train, (0, 2, 1))
        X_valid_seq = np.transpose(sequence_bundle.X_valid, (0, 2, 1))
        X_test_seq = np.transpose(sequence_bundle.X_test, (0, 2, 1))
        y_train_seq = sequence_bundle.y_train
        y_valid_seq = sequence_bundle.y_valid
        y_test_seq = sequence_bundle.y_test

        # Normalise from the training split only, then apply the same scaling to validation/test.
        channel_mean = X_train_seq.mean(axis=(0, 2), keepdims=True)
        channel_std = X_train_seq.std(axis=(0, 2), keepdims=True) + 1e-6
        X_train_seq = (X_train_seq - channel_mean) / channel_std
        X_valid_seq = (X_valid_seq - channel_mean) / channel_std
        X_test_seq = (X_test_seq - channel_mean) / channel_std

        SEQUENCE_CNN_RUN = True
        print(
            {
                "split_strategy": ML_STATE['FIXED_SPLIT_STRATEGY'],
                "fixed_split_block_rows": ML_STATE['FIXED_SPLIT_BLOCK_ROWS'] if ML_STATE['FIXED_SPLIT_STRATEGY'] in {"interleaved_blocks", "episode_aware"} else None,
                "rows_used_train": int(len(sequence_source_frames["train"])),
                "rows_used_validation": int(len(sequence_source_frames["validation"])),
                "rows_used_test": int(len(sequence_source_frames["test"])),
                "windows_total": int(len(X_train_seq) + len(X_valid_seq) + len(X_test_seq)),
                "train_windows": int(len(X_train_seq)),
                "validation_windows": int(len(X_valid_seq)),
                "test_windows": int(len(X_test_seq)),
                "window_size": int(sequence_window_size),
                "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                "feature_set": SEQUENCE_CNN_FEATURE_SET,
                "indicator_channels": sequence_indicator_columns,
                "channel_count": int(len(sequence_input_columns)),
                "sequence_task_mode": sequence_task_mode,
            }
        )

    for _sequence_name in [
        "sequence_train_selection_df",
        "sequence_validation_selection_df",
        "sequence_test_selection_df",
        "sequence_source_frames",
        "sequence_split_frames_full",
        "sequence_split_frames",
        "sequence_bundle",
    ]:
        globals().pop(_sequence_name, None)
    gc.collect()

### Step 2: Build A CNN That Emits One Logit Sequence

The architecture is still a 1D CNN, but the classifier head is now a `1x1` convolution. That lets the network output one class distribution per time step.

In the robust local run, three choices mattered:

- a deeper channel stack (`[64, 128, 256]`) gave the network enough capacity,
- the `1x1` head preserved one prediction per timestamp,
- and class-weighted losses helped the model pay attention to the rarer QC states (the loss is set by `SEQUENCE_CNN_CONFIG["loss"]`).

In [ ]:
if not SEQUENCE_CNN_RUN:
    print("Sequence-labelling CNN model setup skipped.")
else:
    # Seed PyTorch and choose GPU when available.
    torch.manual_seed(ML_STATE['SEED'])
    sequence_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if sequence_device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    # This CNN preserves the time axis: every input timestamp receives an output logit.
    class SequenceLabelCNN(nn.Module):
        def __init__(self, channels: int, output_dim: int) -> None:
            super().__init__()
            # Build a stack of Conv1D blocks with batch norm and GELU activations.
            blocks = []
            in_channels = channels
            kernel_sizes = [7, 5, 3]
            for index, out_channels in enumerate(SEQUENCE_CNN_CONFIG["conv_channels"]):
                kernel_size = kernel_sizes[min(index, len(kernel_sizes) - 1)]
                padding = kernel_size // 2
                blocks.extend(
                    [
                        nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding),
                        nn.BatchNorm1d(out_channels),
                        nn.GELU(),
                        nn.Dropout(SEQUENCE_CNN_CONFIG["dropout"]),
                    ]
                )
                in_channels = out_channels
            self.features = nn.Sequential(*blocks)
            # A 1x1 head converts learned features at each timestamp into class logits.
            self.head = nn.Conv1d(in_channels, output_dim, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.head(self.features(x))

    # Multiclass targets are integer class indexes; binary targets are 0/1 floats for one output logit.
    if sequence_task_mode == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train_seq).long()
        valid_targets_tensor = torch.from_numpy(y_valid_seq).long()
        test_targets_tensor = torch.from_numpy(y_test_seq).long()
        output_dim = len(sequence_class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train_seq.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid_seq.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test_seq.astype(np.float32))
        output_dim = 1

    # Build the loss chosen by SEQUENCE_CNN_CONFIG["loss"]. The table shows each class's training
    # count, its loss weight, and the share of the starting loss it contributes.
    loss_fn, sequence_loss_table = build_classification_loss(
        y_train_seq,
        task_mode=sequence_task_mode,
        class_labels=sequence_class_labels,
        loss_name=SEQUENCE_CNN_CONFIG["loss"],
        focal_gamma=SEQUENCE_CNN_CONFIG["focal_gamma"],
        device=sequence_device,
    )
    print(f"Loss function: {SEQUENCE_CNN_CONFIG['loss']}")
    display(sequence_loss_table)

    # DataLoaders batch windows while preserving the full target sequence for each window.
    train_dataset = TensorDataset(torch.from_numpy(X_train_seq).float(), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(X_valid_seq).float(), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(X_test_seq).float(), test_targets_tensor)

    sequence_loader_kwargs = {}
    num_workers = int(SEQUENCE_CNN_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        sequence_loader_kwargs["num_workers"] = num_workers
        sequence_loader_kwargs["persistent_workers"] = bool(SEQUENCE_CNN_LOADER_CONFIG["persistent_workers"])
        sequence_loader_kwargs["prefetch_factor"] = int(SEQUENCE_CNN_LOADER_CONFIG["prefetch_factor"])
    if SEQUENCE_CNN_LOADER_CONFIG["pin_memory"]:
        sequence_loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(sequence_loader_kwargs.get("pin_memory", False) and sequence_device.type == "cuda")

    train_loader_seq = DataLoader(train_dataset, batch_size=SEQUENCE_CNN_CONFIG["batch_size"], shuffle=True, **sequence_loader_kwargs)
    valid_loader_seq = DataLoader(valid_dataset, batch_size=SEQUENCE_CNN_CONFIG["batch_size"], shuffle=False, **sequence_loader_kwargs)
    test_loader_seq = DataLoader(test_dataset, batch_size=SEQUENCE_CNN_CONFIG["batch_size"], shuffle=False, **sequence_loader_kwargs)

    # Create the model, optimiser, and validation-driven learning-rate scheduler.
    sequence_model = SequenceLabelCNN(channels=len(sequence_input_columns), output_dim=output_dim).to(sequence_device)
    sequence_optimizer = torch.optim.AdamW(
        sequence_model.parameters(),
        lr=SEQUENCE_CNN_CONFIG["learning_rate"],
        weight_decay=SEQUENCE_CNN_CONFIG["weight_decay"],
    )
    sequence_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        sequence_optimizer,
        mode="max",
        factor=SEQUENCE_CNN_CONFIG["lr_decay_factor"],
        patience=SEQUENCE_CNN_CONFIG["lr_patience"],
    )

    # Track the best validation checkpoint separately from the final epoch.
    sequence_best_metric = -np.inf
    sequence_best_epoch = 0
    sequence_patience_counter = 0
    sequence_history = []
    sequence_best_state = None

    # One epoch returns loss plus flattened timestamp predictions for F1 scoring.
    def run_sequence_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        sequence_model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(sequence_device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(sequence_device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = sequence_model(batch_x)
                # CrossEntropyLoss expects logits shaped (batch, classes, time).
                if sequence_task_mode == "multiclass":
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = logits.argmax(dim=1)
                else:
                    logits = logits.squeeze(1)
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    # Standard PyTorch training step: zero gradients, backpropagate, clip, update.
                    sequence_optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if SEQUENCE_CNN_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(sequence_model.parameters(), SEQUENCE_CNN_CONFIG["gradient_clip_norm"])
                    sequence_optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(sequence_device),
            "train_batches": len(train_loader_seq),
            "validation_batches": len(valid_loader_seq),
            "test_batches": len(test_loader_seq),
            "feature_count": len(sequence_input_columns),
            "feature_columns_preview": sequence_input_columns[:8],
        }
    )

### Step 3: Train And Monitor Per-Timestamp Performance

The loop is familiar, but now the validation metric is computed across **all timestamps in the validation windows** rather than one prediction per window.

This is also where we make the usual training choices explicit:

- monitor validation F1 after every epoch,
- keep the best checkpoint,
- stop early when validation no longer improves.

The log also prints validation recall for every class at each epoch, and the early-stopping message says which setting to raise if training stops too soon.

In [ ]:
if not SEQUENCE_CNN_RUN:
    print("Sequence-labelling CNN fit skipped.")
else:
    # Train and monitor validation F1 after every epoch.
    stopped_early = False
    for epoch in range(1, SEQUENCE_CNN_CONFIG["epochs"] + 1):
        # Training updates weights; validation only measures performance.
        train_loss, _, _ = run_sequence_epoch(train_loader_seq, training=True)
        valid_loss, valid_preds, valid_targets = run_sequence_epoch(valid_loader_seq, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(sequence_task_mode), zero_division=0)
        # Recall per class shows whether one class is improving at the expense of another.
        valid_recall = per_class_recall(valid_targets, valid_preds, class_labels=sequence_class_labels)
        # Validation F1 controls both learning-rate decay and checkpoint selection.
        sequence_scheduler.step(valid_f1)
        sequence_history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                **{f"valid_recall_{label}": value for label, value in valid_recall.items()},
                "learning_rate": sequence_optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | "
            f"valid_f1={valid_f1:.4f} | valid_recall={valid_recall}"
        )

        # Save the best model rather than assuming the final epoch is best.
        if valid_f1 > sequence_best_metric + SEQUENCE_CNN_CONFIG["min_delta"]:
            sequence_best_metric = valid_f1
            sequence_best_epoch = epoch
            sequence_patience_counter = 0
            sequence_best_state = copy.deepcopy(sequence_model.state_dict())
            torch.save(
                {
                    "model_state_dict": sequence_best_state,
                    "task_mode": sequence_task_mode,
                    "class_labels": sequence_class_labels,
                    "feature_columns": sequence_input_columns,
                    "window_size": sequence_window_size,
                    "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                    "config": {**SEQUENCE_CNN_CONFIG, **SEQUENCE_CNN_LOADER_CONFIG},
                },
                SEQUENCE_CNN_MODEL_PATH,
            )
        else:
            # Stop when validation F1 has not improved for several epochs.
            sequence_patience_counter += 1
            if sequence_patience_counter >= SEQUENCE_CNN_CONFIG["patience"]:
                stopped_early = True
                print(
                    f"Early stopping at epoch {epoch}: validation F1 has not improved by more than "
                    f"{SEQUENCE_CNN_CONFIG['min_delta']} for {SEQUENCE_CNN_CONFIG['patience']} epochs in a row "
                    f"(best {sequence_best_metric:.4f} at epoch {sequence_best_epoch}). "
                    'To train longer, raise SEQUENCE_CNN_CONFIG["patience"] or SEQUENCE_CNN_CONFIG["epochs"].'
                )
                break

    if not stopped_early:
        print(f"Trained for all {SEQUENCE_CNN_CONFIG['epochs']} epochs (best validation F1 {sequence_best_metric:.4f} at epoch {sequence_best_epoch}).")

### Step 4: Evaluate Per-Timestamp Predictions

This report answers a different question from the earlier CNN section. It tells us how well the model predicts the target label at each timestamp inside the window, not just the single "worst" label for the whole window.

At the end we also compare the CNN result to the saved tree baseline for the same target strategy, so we can see whether the added sequence modelling actually paid off.

In [ ]:
if not SEQUENCE_CNN_RUN:
    print("Sequence-labelling CNN evaluation skipped.")
else:
    # Evaluate the best validation checkpoint on the untouched test windows.
    if sequence_best_state is None:
        raise RuntimeError("Sequence-labelling CNN training did not produce a valid checkpoint")

    sequence_model.load_state_dict(sequence_best_state)
    test_loss, test_preds, test_targets = run_sequence_epoch(test_loader_seq, training=False)
    sequence_test_f1 = f1_score(
        test_targets,
        test_preds,
        average=report_average(sequence_task_mode),
        zero_division=0,
    )

    print(pd.DataFrame(sequence_history).tail())
    print(
        {
            "timestamps_in_test_windows": int(len(test_targets)),
            "device": str(sequence_device),
            "best_validation_f1": float(sequence_best_metric),
            "best_epoch": sequence_best_epoch,
            "test_loss": float(test_loss),
            "test_f1": float(sequence_test_f1),
            "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
            "feature_set": SEQUENCE_CNN_FEATURE_SET,
            "saved_model": str(SEQUENCE_CNN_MODEL_PATH),
        }
    )

    # Build report labels in the same output space the model learned.
    if sequence_task_mode == "multiclass":
        report_labels = list(range(len(sequence_class_labels)))
        report_names = [str(label) for label in sequence_class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    # Plot optimisation history and normalised confusion matrix for timestamp-level errors.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(sequence_history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("Sequence-labelling CNN loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("Per-timestamp CNN confusion matrix")
    plt.tight_layout()
    plt.show()

    # If the tree comparison artifact exists, show whether this CNN improved on the same target strategy.
    strategy_results_path = ML_STATE["ARTIFACT_DIR"] / "extratrees_target_strategy_comparison.csv"
    if strategy_results_path.exists():
        strategy_results = pd.read_csv(strategy_results_path)
        matching_tree_rows = strategy_results[strategy_results["strategy"] == SEQUENCE_CNN_TARGET_STRATEGY]
        if not matching_tree_rows.empty:
            tree_test_f1 = float(matching_tree_rows["test_f1"].iloc[0])
            print(
                {
                    "cnn_test_f1": float(sequence_test_f1),
                    "tree_test_f1_for_same_target": tree_test_f1,
                    "cnn_minus_tree": float(sequence_test_f1 - tree_test_f1),
                }
            )

#### ✏️ Date-Range Demo: Per-Timestamp CNN Predictions on a Specific Interval

This CNN predicts one QC class per timestamp, so its predictions line up directly with the row-level QC task. The selected interval shows true and predicted timestamp-level flags side by side.

The next cell chooses the interval. Leave the start/end as `None` to let the notebook pick about 6 hours of the test split around an issue-flag example, or set your own UTC strings. Either way, the cell prints the exact dates it will use, the dates the test split covers, and example lines you can paste in.

In [ ]:
# ✏️ EDIT: choose the time range for the per-timestamp CNN demo.

# Use the default start/end dates (auto-selected from the test split):
SEQUENCE_CNN_RANGE_START = None
SEQUENCE_CNN_RANGE_END = None

# Or set specific UTC strings (pick dates inside the test-split span printed below):
# SEQUENCE_CNN_RANGE_START = "2026-02-10 00:00:00Z"
# SEQUENCE_CNN_RANGE_END = "2026-02-11 00:00:00Z"

SEQUENCE_CNN_MAX_POINTS_TO_PLOT = 1024  # Plot at most this many points so the figure stays readable.

# --- No edits needed below this line ---
# Resolve the range now so the exact dates are visible before the demo runs.
sequence_range_selection = select_time_range(
    ML_STATE['test_df'],
    time_column="Time UTC",
    label_column=ML_STATE['TARGET_FLAG'],
    start=SEQUENCE_CNN_RANGE_START,
    end=SEQUENCE_CNN_RANGE_END,
    max_points=SEQUENCE_CNN_MAX_POINTS_TO_PLOT,
)
print(
    describe_time_range_selection(
        sequence_range_selection,
        reference_frame=ML_STATE['test_df'],
        variable_prefix="SEQUENCE_CNN",
        label_meanings=ML_STATE['FLAG_EXAMPLE_LABEL_MEANINGS'],
    )
)

In [ ]:
if not SEQUENCE_CNN_RUN:
    print("Per-timestamp CNN demo skipped because the model was not trained in this run.")
else:
    # Load only the row-level data needed for the selected interval.
    sequence_interval_rows = load_rows_for_time_range(
        ML_STATE['metadata'],
        row_cache_dir=Path(ML_STATE['ROW_CACHE_DIR']),
        start=sequence_range_selection["start"],
        end=sequence_range_selection["end"],
        columns=ML_STATE['ROW_USE_COLUMNS'],
    )

    if sequence_interval_rows.empty:
        print("No row-level data was found in the requested sequence-labelling CNN range.")
    else:
        # Rebuild the same engineered features used during sequence-CNN training.
        sequence_interval_feature_df = sequence_interval_rows.copy()
        # Delta features highlight abrupt local changes in the selected interval.
        for column in ML_STATE['measurement_columns']:
            sequence_interval_feature_df[f"{column} abs_delta"] = sequence_interval_feature_df[column].diff().abs().fillna(0.0)

        # Rolling context features must be recreated before the model can score the interval.
        sequence_context_source = sequence_interval_feature_df[["Time UTC", *ML_STATE['measurement_columns']]].copy()
        sequence_context_frame, _ = add_temporal_context_features(
            sequence_context_source,
            measurement_columns=ML_STATE['measurement_columns'],
            lag_steps=(),
            rolling_windows=(5,),
        )
        for column in sequence_context_columns:
            sequence_interval_feature_df[column] = sequence_context_frame[column]

        # Recreate cyclical time features with the same formula used in training.
        radians_hour = 2.0 * np.pi * sequence_interval_feature_df["Time UTC"].dt.hour / 24.0
        radians_doy = 2.0 * np.pi * sequence_interval_feature_df["Time UTC"].dt.dayofyear / 366.0
        sequence_interval_feature_df["hour_sin"] = np.sin(radians_hour)
        sequence_interval_feature_df["hour_cos"] = np.cos(radians_hour)
        sequence_interval_feature_df["doy_sin"] = np.sin(radians_doy)
        sequence_interval_feature_df["doy_cos"] = np.cos(radians_doy)

        # Add the same optional range-check channels used in training.
        sequence_interval_feature_df, _ = add_indicator_channels(
            sequence_interval_feature_df,
            SEQUENCE_CNN_CONFIG["indicator_channels"],
        )

        # Apply the same target strategy so plotted labels match the trained model output.
        sequence_interval_target_df, _, _ = apply_target_strategy(
            sequence_interval_feature_df.dropna(subset=[ML_STATE['TARGET_FLAG']]).copy(),
            ML_STATE['TARGET_FLAG'],
            SEQUENCE_CNN_TARGET_STRATEGY,
            issue_labels=ML_STATE['ISSUE_LABELS'],
        )
        sequence_interval_target_df = sequence_interval_target_df.rename(columns={"strategy_target": "model_target"})
        sequence_interval_target_df = sequence_interval_target_df[
            (sequence_interval_target_df["Time UTC"] >= sequence_range_selection["start"])
            & (sequence_interval_target_df["Time UTC"] <= sequence_range_selection["end"])
        ].reset_index(drop=True)

        # Convert the interval into complete windows before running the CNN.
        sequence_interval_bundle = build_sequence_label_interval_data(
            sequence_interval_target_df,
            feature_columns=sequence_input_columns,
            target_column="model_target",
            window_size=sequence_window_size,
        )

        if len(sequence_interval_bundle["raw_sequences"]) == 0:
            print("The selected interval is shorter than one full sequence-labelling window after preprocessing, so the demo is skipped.")
        else:
            # Use training-split normalisation stored in channel_mean/channel_std during prediction.
            sequence_predicted_labels = predict_sequence_label_cnn(
                sequence_model,
                sequence_interval_bundle["raw_sequences"],
                task_mode=sequence_task_mode,
                class_labels=sequence_class_labels,
                device=str(sequence_device),
                channel_mean=channel_mean,
                channel_std=channel_std,
                batch_size=SEQUENCE_CNN_CONFIG["batch_size"],
            )

            # Flatten window outputs so metrics and interval bands are calculated per timestamp.
            sequence_flat_predictions = sequence_predicted_labels.reshape(-1)
            sequence_flat_targets = sequence_interval_bundle["raw_targets"].reshape(-1)
            sequence_flat_times = pd.to_datetime(sequence_interval_bundle["raw_times"].reshape(-1), utc=True)
            sequence_plot_frame = pd.DataFrame(
                {
                    "Time UTC": sequence_flat_times,
                    "true_label": sequence_flat_targets,
                    "predicted_label": sequence_flat_predictions,
                }
            )
            sequence_interval_origin = infer_interval_origin(
                sequence_range_selection["start"],
                sequence_range_selection["end"],
                {"train": ML_STATE['train_df'], "validation": ML_STATE['valid_df'], "test": ML_STATE['test_df']},
            )
            sequence_interval_metrics = compute_interval_classification_metrics(
                sequence_flat_targets,
                sequence_flat_predictions,
                labels=sequence_class_labels,
                average=report_average(sequence_task_mode),
                target_names=[str(label) for label in sequence_class_labels],
            )
            sequence_plot_palette = DEFAULT_FLAG_PALETTE if sequence_task_mode == "multiclass" else {0: "#1f77b4", 1: "#d62728"}
            # Convert adjacent labels into time intervals to make the overlay plot readable.
            sequence_true_intervals = build_labeled_intervals(
                sequence_plot_frame,
                time_column="Time UTC",
                label_column="true_label",
            )
            sequence_pred_intervals = build_labeled_intervals(
                sequence_plot_frame,
                time_column="Time UTC",
                label_column="predicted_label",
            )

            print(
                {
                    "interval_origin": sequence_interval_origin,
                    "timestamps_in_interval": int(len(sequence_plot_frame)),
                    "interval_f1": sequence_interval_metrics["f1"],
                    "target_strategy": SEQUENCE_CNN_TARGET_STRATEGY,
                    "feature_set": SEQUENCE_CNN_FEATURE_SET,
                }
            )
            print(sequence_interval_metrics["report_text"])
            display(
                pd.DataFrame(
                    {
                        "true_count": pd.Series(sequence_flat_targets).value_counts().sort_index(),
                        "predicted_count": pd.Series(sequence_flat_predictions).value_counts().sort_index(),
                    }
                ).fillna(0).astype(int)
            )

            sequence_demo_figure = plot_time_series_with_bands(
                sequence_interval_target_df,
                band_specs=[
                    {"title": f"True timestamp {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']}", "intervals": sequence_true_intervals, "palette": sequence_plot_palette},
                    {"title": f"Sequence CNN {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']} prediction", "intervals": sequence_pred_intervals, "palette": sequence_plot_palette},
                ],
                measurement_column=ML_STATE['PLOT_MEASUREMENT_COLUMN'],
                secondary_column=ML_STATE['PLOT_SECONDARY_COLUMN'],
                max_points=SEQUENCE_CNN_MAX_POINTS_TO_PLOT,
                title="Sequence-labelling CNN predictions on a selected time range",
                label_meanings=ML_STATE['FLAG_EXAMPLE_LABEL_MEANINGS'],
                legend_title=ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME'],
            )
            plt.show()

            sequence_cm_fig, sequence_cm_ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                confusion_matrix=sequence_interval_metrics["confusion_matrix"],
                display_labels=sequence_interval_metrics["display_labels"],
            ).plot(ax=sequence_cm_ax, cmap="Blues", colorbar=False)
            sequence_cm_ax.set_title("Sequence-labelling CNN confusion matrix on the selected range")
            plt.tight_layout()
            plt.show()

### Reflection Questions: Sequence-labelling CNN

1. Where does predicting one flag per timestamp clearly help more than the earlier window-classification CNN?
2. Do the remaining mistakes look like a feature-set problem, a target-strategy problem, or a pure optimisation problem?
3. If you had to choose one next experiment for this model, would you change the input features, the target strategy, or the sequence length?

Add your own notes or follow-up questions here:

- 
- 
- 

## Part 3 — Loss Functions For Rare Labels

QC issue flags are rare, so the loss function decides how much the model cares about them. The machine-learning notebook introduced four `loss` options. This part trains the per-timestamp CNN from Part 2 once per option, on exactly the same windows, architecture, and random seed, so the loss is the only thing that changes.

| `loss` | Class weighting | What to watch for |
|---|---|---|
| `"weighted_cross_entropy"` | Each class weighted by `total / (n_classes × class_count)`, so every class contributes the same share of the loss | Finds more rare-class rows, usually at the cost of precision |
| `"sqrt_weighted_cross_entropy"` | The square root of those weights | A middle ground: rare classes still count more, but less aggressively |
| `"cross_entropy"` | None | Can ignore a rare class entirely, because the common class dominates the loss |
| `"focal"` | None, but examples the model already gets right are down-weighted | Concentrates training on hard examples without fixed class weights |

Read the results table per class, not only by macro F1: one option can raise recall for a rare flag while lowering its precision. If every option fails on the same class, the problem is usually the inputs or the target definition rather than the loss.

### ✏️ Loss Comparison Settings

The next cell chooses which loss options to compare and how many epochs each one trains. Every run keeps its best validation epoch, then reports test results for each class.

In [ ]:
# ✏️ EDIT: which loss options to compare, and how long each one trains.
LOSS_COMPARISON_OPTIONS = ["weighted_cross_entropy", "sqrt_weighted_cross_entropy", "cross_entropy", "focal"]
LOSS_COMPARISON_EPOCHS = 4  # Each option trains for this many epochs; the best validation epoch is kept.
LOSS_COMPARISON_FOCAL_GAMMA = 2.0  # Only used by "focal".

### Train One CNN Per Loss Option

The next cell reuses the per-timestamp windows, data loaders, and `SequenceLabelCNN` architecture from Part 2. For each loss option it starts from the same random seed, trains, keeps the epoch with the best validation F1, and scores the test windows. The table at the end has one row per loss option: how much of the starting loss comes from the rarest class (`rare_class_loss_share`), the validation F1 used to pick the epoch, and test recall and precision for every class.

In [ ]:
def predict_sequence_split(model, loader):
    """Return flattened true and predicted class indexes for one data loader."""
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            logits = model(batch_x.to(sequence_device))
            if sequence_task_mode == "multiclass":
                batch_predictions = logits.argmax(dim=1)
            else:
                batch_predictions = (torch.sigmoid(logits.squeeze(1)) >= 0.5).long()
            predictions.append(batch_predictions.cpu().reshape(-1).numpy())
            targets.append(batch_y.reshape(-1).numpy())
    return np.concatenate(targets).astype(int), np.concatenate(predictions).astype(int)


if not SEQUENCE_CNN_RUN:
    print("Loss comparison skipped because the per-timestamp CNN data was not built in this run.")
else:
    comparison_output_dim = len(sequence_class_labels) if sequence_task_mode == "multiclass" else 1
    report_indexes = list(range(len(sequence_class_labels))) if sequence_task_mode == "multiclass" else [0, 1]
    report_names = [str(label) for label in sequence_class_labels] if sequence_task_mode == "multiclass" else ["0", "1"]
    loss_comparison_rows = []

    for loss_name in LOSS_COMPARISON_OPTIONS:
        # Same data, same architecture, same seed: only the loss changes.
        torch.manual_seed(ML_STATE['SEED'])
        comparison_model = SequenceLabelCNN(channels=len(sequence_input_columns), output_dim=comparison_output_dim).to(sequence_device)
        comparison_loss_fn, comparison_weight_table = build_classification_loss(
            y_train_seq,
            task_mode=sequence_task_mode,
            class_labels=sequence_class_labels,
            loss_name=loss_name,
            focal_gamma=LOSS_COMPARISON_FOCAL_GAMMA,
            device=sequence_device,
        )
        comparison_optimizer = torch.optim.AdamW(
            comparison_model.parameters(),
            lr=SEQUENCE_CNN_CONFIG["learning_rate"],
            weight_decay=SEQUENCE_CNN_CONFIG["weight_decay"],
        )

        best_valid_f1, best_epoch, best_state = -1.0, 0, None
        for epoch in range(1, LOSS_COMPARISON_EPOCHS + 1):
            comparison_model.train()
            for batch_x, batch_y in train_loader_seq:
                batch_x = batch_x.to(sequence_device)
                batch_y = batch_y.to(sequence_device)
                logits = comparison_model(batch_x)
                if sequence_task_mode != "multiclass":
                    logits = logits.squeeze(1)
                loss = comparison_loss_fn(logits, batch_y)
                comparison_optimizer.zero_grad(set_to_none=True)
                loss.backward()
                comparison_optimizer.step()
            valid_true, valid_pred = predict_sequence_split(comparison_model, valid_loader_seq)
            valid_f1 = f1_score(valid_true, valid_pred, average=report_average(sequence_task_mode), zero_division=0)
            if valid_f1 > best_valid_f1:
                best_valid_f1, best_epoch, best_state = valid_f1, epoch, copy.deepcopy(comparison_model.state_dict())

        comparison_model.load_state_dict(best_state)
        test_true, test_pred = predict_sequence_split(comparison_model, test_loader_seq)
        test_report = classification_report(
            test_true,
            test_pred,
            labels=report_indexes,
            target_names=report_names,
            output_dict=True,
            zero_division=0,
        )
        result_row = {
            "loss": loss_name,
            "best_epoch": best_epoch,
            # Share of the starting loss that comes from the rarest training class under this loss.
            "rare_class_loss_share": float(
                comparison_weight_table.loc[comparison_weight_table["train_count"].idxmin(), "share_of_starting_loss"]
            ),
            "valid_f1": round(float(best_valid_f1), 3),
            "test_macro_f1": round(float(test_report["macro avg"]["f1-score"]), 3),
        }
        for name in report_names:
            result_row[f"test_recall_{name}"] = round(float(test_report[name]["recall"]), 3)
            result_row[f"test_precision_{name}"] = round(float(test_report[name]["precision"]), 3)
        loss_comparison_rows.append(result_row)
        print(f"Finished {loss_name}: best epoch {best_epoch}, test macro F1 {result_row['test_macro_f1']}")

    loss_comparison_table = pd.DataFrame(loss_comparison_rows).set_index("loss")
    display(loss_comparison_table)

### Reflection Questions: Loss Functions

1. Which loss gave the best balance between recall and precision for the rarest class? Is that the same option that gave the best macro F1?
2. Did any option collapse to predicting only the common class? What does its `rare_class_loss_share` (the share of the starting loss that comes from the rarest class) suggest about why?
3. If no loss option helps a particular class, would you change the input features, the target definition, or the amount of training data first?

Add your own notes or follow-up questions here:

-
-
-

## Part 4 — Transformer: Forecast Good Data, Then Flag Large Errors

The machine-learning notebook introduced the small transformer encoder and used it to classify QC labels directly. This part reuses the same encoder ingredients for a different job: learning what normal sensor behaviour looks like.

### ✏️ Transformer Variant: Forecast Good Data, Then Flag Large Errors

The supervised transformer in the machine-learning notebook learns QC labels directly. A different anomaly-detection pattern is to train only on good data, ask the model to predict the next sensor value from recent history, and treat large prediction errors as suspicious.

This is useful when labels are sparse or when you want to catch failure modes that were not well represented in the reviewed labels. The model is not learning what each bad flag looks like. It is learning what normal behaviour usually looks like.

The training setup is:

1. build contiguous windows where the previous `lookback` rows predict the next `target_column` value,
2. train on good-only windows so the model learns normal behaviour,
3. validate on good-only windows for early stopping,
4. score validation/test windows that include both good and issue labels,
5. flag windows whose standardised next-value prediction error is above a good-validation threshold.

`include_target_history` controls whether the model sees previous values of the same column it is trying to predict. If `target_column` is `cond_value_ctd`, then `include_target_history=True` lets the model use previous conductivity values to predict the next conductivity value.

`include_target_history=False` makes the model **exogenous** for the target: it predicts the target from the other sensor/context columns, not from the target's own recent history. This can help for persistent failures such as conductivity plugs, because bad conductivity can be easy to predict from previous bad conductivity. Exogenous prediction asks a cleaner question: *given the other sensors, what should conductivity have been?*

| Parameter | What it changes |
|---|---|
| `target_column` | measurement value to predict one step ahead |
| `lookback` | number of previous rows used as transformer context |
| `include_target_history` | `True` uses the target's own recent history; `False` uses only other columns as exogenous predictors |
| `max_gap_seconds` | drops windows that jump across large time gaps |
| `max_train_source_rows` / `max_eval_source_rows` | caps labelled rows loaded before windows are built |
| `max_train_windows` / `max_eval_windows` | caps final transformer examples after valid windows are found |
| `loss` | `mse` penalizes large errors strongly; `huber` is more robust to occasional large deviations |
| `threshold_quantile` | good-validation error percentile used as the anomaly cutoff |

This section follows the same broad idea used in recent transformer anomaly-detection and forecasting work:

- [Anomaly Transformer](https://arxiv.org/abs/2110.02642) frames unsupervised time-series anomaly detection around how abnormal points relate to the rest of the series.
- [TranAD](https://arxiv.org/abs/2201.07284) uses transformer sequence encoders for multivariate time-series anomaly detection and diagnosis.
- [PatchTST](https://arxiv.org/abs/2211.14730) is a strong example of transformer-based time-series forecasting, especially the idea that forecasting can be a useful self-supervised signal.

The implementation below keeps the idea simple: predict the next value of one measurement column, set an error threshold from good validation windows, then inspect which validation/test points exceed that threshold.

In [ ]:
# ✏️ EDIT: forecasting-transformer settings.
# Participant choice: keep this False for an exogenous detector, or True to let
# the model use the target's own recent history. Conductivity plugs default to
# False because persistent bad conductivity can otherwise become easy to predict.
FORECAST_INCLUDE_TARGET_HISTORY = ML_STATE['DATASET_PROFILE_ID'] != "conductivity_plugs"

# Huber is a robust regression loss. MSE is also useful when large prediction
# errors should be penalized more aggressively.
FORECAST_LOSS = "huber" if ML_STATE['DATASET_PROFILE_ID'] == "conductivity_plugs" else "mse"

# The threshold is set from good-validation errors. Lower values catch more
# potential anomalies; higher values are more conservative.
FORECAST_THRESHOLD_QUANTILE = 0.95 if ML_STATE['DATASET_PROFILE_ID'] == "conductivity_plugs" else 0.99

FORECAST_TRANSFORMER_CONFIG = {
    # Predict the next value of this column. For conductivity plugs this is cond_value_ctd.
    "target_column": ML_STATE['PLOT_MEASUREMENT_COLUMN'],
    # Use recent rows as context. Larger values see longer trends but cost more memory.
    "lookback": 96,
    # Windows are sampled after contiguous candidate windows are built, so sequence order is preserved.
    "max_train_windows": 100_000 if ML_STATE['DATASET_PROFILE_ID'] == "conductivity_plugs" else 20_000,
    "max_eval_windows": 20_000,
    # These cap the labelled source rows loaded before windows are built.
    # The helper keeps contiguous blocks so a 96-row history is still a real local sequence.
    "max_train_source_rows": 300_000 if ML_STATE['DATASET_PROFILE_ID'] == "conductivity_plugs" else 120_000,
    "max_eval_source_rows": 120_000,
    # Larger blocks preserve longer local stretches but may load more rows than needed.
    "context_block_rows": 4096,
    # Reject windows that cross large time gaps. This keeps "next row" close to "next moment".
    "max_gap_seconds": 5.0,
    # True: use previous target values as features. False: use only exogenous/context features.
    "include_target_history": FORECAST_INCLUDE_TARGET_HISTORY,
    # d_model is the transformer embedding size: larger can learn richer patterns but costs more memory.
    "d_model": 64,
    # nhead is the number of attention heads. d_model must be divisible by nhead.
    "nhead": 4,
    # num_layers is the number of stacked transformer encoder blocks.
    "num_layers": 2,
    # dim_feedforward controls the hidden width inside each transformer block.
    "dim_feedforward": 128,
    # Dropout randomly masks activations during training to reduce overfitting.
    "dropout": 0.15,
    # epochs is the maximum pass count; early stopping can stop sooner.
    "epochs": 6,
    # batch_size controls examples per optimizer step. Larger is faster on GPU but uses more memory.
    "batch_size": 256,
    # learning_rate controls optimizer step size.
    "learning_rate": 1e-3,
    # weight_decay is mild L2 regularization on model weights.
    "weight_decay": 1e-4,
    # loss can be "mse" or "huber".
    "loss": FORECAST_LOSS,
    # patience and min_delta control early stopping on good-validation loss.
    "patience": 2,
    "min_delta": 1e-4,
    # Gradient clipping prevents rare large updates from destabilizing training.
    "gradient_clip_norm": 1.0,
    # A 95th-percentile good-validation threshold worked better for conductivity plugs locally.
    "threshold_quantile": FORECAST_THRESHOLD_QUANTILE,
}

# DataLoader settings. num_workers=0 is slower but safest inside shared notebook environments.
FORECAST_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": bool(torch.cuda.is_available()),
    "persistent_workers": False,
    "prefetch_factor": 2,
}

# --- No edits needed below this line ---
display(pd.Series(FORECAST_TRANSFORMER_CONFIG, name="value").rename_axis("forecast_parameter").to_frame())
print(
    {
        "training_data": "good reviewed rows only",
        "anomaly_score": "absolute standardised next-value prediction error",
        "feature_history": "target included" if FORECAST_TRANSFORMER_CONFIG["include_target_history"] else "target excluded",
    }
)

#### Build Good-Only Forecasting Windows

Each training example contains `lookback` previous rows and one next value to predict.

For training, the context rows and the next target row must all have good QC/custom labels. The validation-loss windows are also good-only, because they control early stopping for a normal-behaviour model. For validation/test scoring, we keep both good and issue rows so prediction error can be compared against the reviewed labels.

In [ ]:
def select_evenly_spaced_positions(values: np.ndarray, limit: int | None) -> np.ndarray:
    """Keep positions spread across time when a split has more candidate windows than we want."""
    if limit is None or len(values) <= limit:
        return values
    positions = np.linspace(0, len(values) - 1, num=int(limit), dtype=int)
    return values[positions]


def build_next_value_forecast_windows(
    frame: pd.DataFrame,
    *,
    feature_columns: list[str],
    target_column: str,
    label_column: str,
    good_labels: list[int] | tuple[int, ...],
    issue_labels: list[int] | tuple[int, ...],
    lookback: int,
    max_windows: int | None,
    require_good_window: bool,
    max_gap_seconds: float | None,
) -> dict[str, object]:
    """Build transformer windows for next-value forecasting.

    `require_good_window=True` is used for training and validation loss so the
    model learns normal dynamics only. `max_gap_seconds` prevents windows from
    jumping across large time gaps and treating distant rows as adjacent.
    """
    if not feature_columns:
        raise ValueError("feature_columns must contain at least one column.")
    if lookback < 2:
        raise ValueError("lookback must be at least 2")

    work_columns = list(dict.fromkeys(["Time UTC", "source_file", label_column, target_column, *feature_columns]))
    work = frame[work_columns].copy()
    work["Time UTC"] = pd.to_datetime(work["Time UTC"], utc=True)
    for column in [label_column, target_column, *feature_columns]:
        work[column] = pd.to_numeric(work[column], errors="coerce")
    work = work.dropna(subset=["Time UTC", "source_file", label_column, target_column, *feature_columns]).copy()
    if work.empty:
        return {
            "X": np.empty((0, lookback, len(feature_columns)), dtype=np.float32),
            "y": np.empty((0,), dtype=np.float32),
            "metadata": pd.DataFrame(columns=["Time UTC", "source_file", "target_value", "qc_label", "is_good", "is_issue"]),
        }

    source_groups = list(work.groupby("source_file", sort=False, observed=False))
    per_source_limit = None if max_windows is None else max(1, int(np.ceil(int(max_windows) / max(len(source_groups), 1))))
    good_label_set = {int(label) for label in good_labels}
    issue_label_set = {int(label) for label in issue_labels}

    X_parts = []
    y_values = []
    metadata_rows = []

    for source_file, source_frame in source_groups:
        source_frame = source_frame.sort_values("Time UTC").reset_index(drop=True)
        if len(source_frame) <= lookback:
            continue

        values = source_frame[feature_columns].to_numpy(dtype=np.float32)
        target_values = source_frame[target_column].to_numpy(dtype=np.float32)
        labels = source_frame[label_column].astype(int).to_numpy()
        time_values = source_frame["Time UTC"].astype("int64").to_numpy(copy=False)
        finite_rows = np.isfinite(values).all(axis=1) & np.isfinite(target_values)
        good_rows = np.isin(labels, list(good_label_set))
        max_gap_ns = None if max_gap_seconds is None else int(max_gap_seconds * 1_000_000_000)

        candidate_ends = []
        for end_index in range(lookback, len(source_frame)):
            context_slice = slice(end_index - lookback, end_index)
            if not finite_rows[context_slice].all() or not finite_rows[end_index]:
                continue
            if max_gap_ns is not None:
                window_times = time_values[end_index - lookback : end_index + 1]
                if np.diff(window_times).max() > max_gap_ns:
                    continue
            if require_good_window and (not good_rows[context_slice].all() or not good_rows[end_index]):
                continue
            candidate_ends.append(end_index)

        selected_ends = select_evenly_spaced_positions(np.asarray(candidate_ends, dtype=int), per_source_limit)
        for end_index in selected_ends.tolist():
            context_slice = slice(end_index - lookback, end_index)
            X_parts.append(values[context_slice])
            y_values.append(float(target_values[end_index]))
            qc_label = int(labels[end_index])
            metadata_rows.append(
                {
                    "Time UTC": source_frame.loc[end_index, "Time UTC"],
                    "source_file": source_file,
                    "target_value": float(target_values[end_index]),
                    "qc_label": qc_label,
                    "is_good": qc_label in good_label_set,
                    "is_issue": qc_label in issue_label_set,
                }
            )

    if not X_parts:
        return {
            "X": np.empty((0, lookback, len(feature_columns)), dtype=np.float32),
            "y": np.empty((0,), dtype=np.float32),
            "metadata": pd.DataFrame(columns=["Time UTC", "source_file", "target_value", "qc_label", "is_good", "is_issue"]),
        }

    X = np.stack(X_parts).astype(np.float32, copy=False)
    y = np.asarray(y_values, dtype=np.float32)
    metadata_frame = pd.DataFrame(metadata_rows)

    if max_windows is not None and len(X) > int(max_windows):
        keep = select_evenly_spaced_positions(np.arange(len(X)), int(max_windows))
        X = X[keep]
        y = y[keep]
        metadata_frame = metadata_frame.iloc[keep].reset_index(drop=True)
    else:
        metadata_frame = metadata_frame.reset_index(drop=True)

    return {"X": X, "y": y, "metadata": metadata_frame}


def select_contiguous_time_blocks(
    frame: pd.DataFrame,
    *,
    max_rows: int | None,
    block_rows: int,
    lookback: int,
    time_column: str = "Time UTC",
    source_column: str = "source_file",
) -> pd.DataFrame:
    """Keep a row budget as local time blocks instead of isolated sampled rows.

    Forecasting windows need adjacent rows. Evenly sampling individual rows can
    turn a 96-row history into 96 distant timestamps, so this helper samples
    whole contiguous blocks spread across source files and time.
    """
    if max_rows is None or len(frame) <= int(max_rows):
        return frame.sort_values([source_column, time_column]).reset_index(drop=True)

    block_rows = max(int(block_rows), int(lookback) + 1)
    candidate_blocks = []
    for source_file, source_frame in frame.groupby(source_column, sort=False, observed=False):
        source_frame = source_frame.sort_values(time_column).reset_index(drop=True)
        for start in range(0, len(source_frame), block_rows):
            stop = min(start + block_rows, len(source_frame))
            if stop - start >= int(lookback) + 1:
                candidate_blocks.append(source_frame.iloc[start:stop])

    if not candidate_blocks:
        return frame.iloc[0:0].copy()
    block_limit = max(1, int(np.ceil(int(max_rows) / block_rows)))
    selected_block_indices = select_evenly_spaced_positions(np.arange(len(candidate_blocks)), block_limit)
    selected = pd.concat([candidate_blocks[index] for index in selected_block_indices], ignore_index=True)
    if len(selected) > int(max_rows):
        selected = selected.iloc[: int(max_rows)]
    return selected.sort_values([source_column, time_column]).reset_index(drop=True)


if not PYTORCH_READY:
    FORECAST_TRANSFORMER_RUN = False
    print("Good-data forecasting transformer skipped because PyTorch is not available in the current notebook state.")
else:
    # The target is predicted one row into the future. It can be included or excluded from the input history.
    forecast_target_column = FORECAST_TRANSFORMER_CONFIG["target_column"]
    if forecast_target_column not in ML_STATE['ROW_USE_COLUMNS']:
        raise ValueError(f"Forecast target {forecast_target_column!r} is not in the active row columns.")
    # Start with all measurement columns available in the row-level cache.
    forecast_feature_columns = [column for column in ML_STATE['measurement_columns'] if column in ML_STATE['ROW_USE_COLUMNS']]
    if not FORECAST_TRANSFORMER_CONFIG["include_target_history"]:
        # Exogenous mode: remove the target from the input features, but still predict it as y.
        forecast_feature_columns = [column for column in forecast_feature_columns if column != forecast_target_column]
    if forecast_target_column not in forecast_feature_columns:
        if FORECAST_TRANSFORMER_CONFIG["include_target_history"]:
            raise ValueError(f"Forecast target {forecast_target_column!r} is not in the active measurement columns.")
    if not forecast_feature_columns:
        raise ValueError("No forecast feature columns remain after applying include_target_history.")

    # Use the full reviewed label table, not the DATA_FRACTION sample, so contiguous forecast windows stay realistic.
    forecast_label_df = ML_STATE['reviewed_label_df']
    forecast_split_block_rows = ML_STATE['FIXED_SPLIT_BLOCK_ROWS'] if ML_STATE['FIXED_SPLIT_BLOCK_ROWS'] is not None else ML_STATE['INTERLEAVED_BLOCK_ROWS']
    # Rebuild the same train/validation/test split on the label table used for forecasting.
    forecast_label_splits = split_frame_by_strategy(
        forecast_label_df,
        train_fraction=ML_STATE['TRAIN_FRACTION'],
        validation_fraction=ML_STATE['VALIDATION_FRACTION'],
        strategy=ML_STATE['FIXED_SPLIT_STRATEGY'],
        source_column="source_file",
        block_rows=forecast_split_block_rows,
        issue_column="issue",
        target_column=ML_STATE['TARGET_FLAG'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        episode_context_rows=ML_STATE['EPISODE_CONTEXT_ROWS'],
        episode_merge_gap_rows=ML_STATE['EPISODE_MERGE_GAP_ROWS'],
        purge_gap_rows=ML_STATE['EPISODE_PURGE_GAP_ROWS'],
    )
    # Row budgets are applied as contiguous blocks, not isolated rows, because sequence models need neighbours.
    forecast_selection_frames = {
        "train": select_contiguous_time_blocks(
            forecast_label_splits["train"],
            max_rows=FORECAST_TRANSFORMER_CONFIG["max_train_source_rows"],
            block_rows=FORECAST_TRANSFORMER_CONFIG["context_block_rows"],
            lookback=FORECAST_TRANSFORMER_CONFIG["lookback"],
        ),
        "validation": select_contiguous_time_blocks(
            forecast_label_splits["validation"],
            max_rows=FORECAST_TRANSFORMER_CONFIG["max_eval_source_rows"],
            block_rows=FORECAST_TRANSFORMER_CONFIG["context_block_rows"],
            lookback=FORECAST_TRANSFORMER_CONFIG["lookback"],
        ),
        "test": select_contiguous_time_blocks(
            forecast_label_splits["test"],
            max_rows=FORECAST_TRANSFORMER_CONFIG["max_eval_source_rows"],
            block_rows=FORECAST_TRANSFORMER_CONFIG["context_block_rows"],
            lookback=FORECAST_TRANSFORMER_CONFIG["lookback"],
        ),
    }
    # Load the measurement columns for only the selected forecast rows from parquet.
    forecast_split_rows = materialize_reviewed_split_frames(
        ML_STATE['selected_paths'],
        forecast_selection_frames,
        columns=ML_STATE['ROW_USE_COLUMNS'],
        target_flag=ML_STATE['TARGET_FLAG'],
        task_mode=ML_STATE['task_mode'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
    )
    lookback = int(FORECAST_TRANSFORMER_CONFIG["lookback"])
    # Training windows are good-only: the model should learn normal next-value behaviour.
    forecast_train_bundle = build_next_value_forecast_windows(
        forecast_split_rows["train"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=ML_STATE['TARGET_FLAG'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_train_windows"],
        require_good_window=True,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )
    # Good-only validation controls early stopping without learning from known issue windows.
    forecast_valid_good_bundle = build_next_value_forecast_windows(
        forecast_split_rows["validation"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=ML_STATE['TARGET_FLAG'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_eval_windows"],
        require_good_window=True,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )
    # Scoring windows include good and issue labels so we can compare error against reviewed labels.
    forecast_valid_bundle = build_next_value_forecast_windows(
        forecast_split_rows["validation"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=ML_STATE['TARGET_FLAG'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_eval_windows"],
        require_good_window=False,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )
    forecast_test_bundle = build_next_value_forecast_windows(
        forecast_split_rows["test"],
        feature_columns=forecast_feature_columns,
        target_column=forecast_target_column,
        label_column=ML_STATE['TARGET_FLAG'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        lookback=lookback,
        max_windows=FORECAST_TRANSFORMER_CONFIG["max_eval_windows"],
        require_good_window=False,
        max_gap_seconds=FORECAST_TRANSFORMER_CONFIG["max_gap_seconds"],
    )

    if (
        len(forecast_train_bundle["X"]) == 0
        or len(forecast_valid_good_bundle["X"]) == 0
        or len(forecast_valid_bundle["X"]) == 0
        or len(forecast_test_bundle["X"]) == 0
    ):
        FORECAST_TRANSFORMER_RUN = False
        print(
            {
                "skip_reason": "At least one split produced no next-value forecasting windows.",
                "train_windows": int(len(forecast_train_bundle["X"])),
                "validation_good_windows": int(len(forecast_valid_good_bundle["X"])),
                "validation_windows": int(len(forecast_valid_bundle["X"])),
                "test_windows": int(len(forecast_test_bundle["X"])),
            }
        )
    else:
        forecast_feature_mean = forecast_train_bundle["X"].mean(axis=(0, 1), keepdims=True)
        forecast_feature_std = forecast_train_bundle["X"].std(axis=(0, 1), keepdims=True) + 1e-6
        forecast_target_mean = float(forecast_train_bundle["y"].mean())
        forecast_target_std = float(forecast_train_bundle["y"].std() + 1e-6)

        for bundle in [forecast_train_bundle, forecast_valid_good_bundle, forecast_valid_bundle, forecast_test_bundle]:
            bundle["X_scaled"] = ((bundle["X"] - forecast_feature_mean) / forecast_feature_std).astype(np.float32, copy=False)
            bundle["y_scaled"] = ((bundle["y"] - forecast_target_mean) / forecast_target_std).astype(np.float32, copy=False)

        FORECAST_TRANSFORMER_RUN = True
        print(
            {
                "target_column": forecast_target_column,
                "lookback": lookback,
                "features_per_step": len(forecast_feature_columns),
                "train_source_rows_loaded": int(len(forecast_selection_frames["train"])),
                "validation_source_rows_loaded": int(len(forecast_selection_frames["validation"])),
                "test_source_rows_loaded": int(len(forecast_selection_frames["test"])),
                "train_good_windows": int(len(forecast_train_bundle["X"])),
                "validation_good_windows": int(len(forecast_valid_good_bundle["X"])),
                "validation_scoring_windows": int(len(forecast_valid_bundle["X"])),
                "test_scoring_windows": int(len(forecast_test_bundle["X"])),
            }
        )

#### Train A Next-Value Transformer On Good Windows

This model has the same encoder ingredients as the classifier in the machine-learning notebook: input projection, positional embedding, transformer encoder blocks, and a small prediction head.

The difference is the loss. Instead of cross-entropy against QC labels, this model minimizes a regression loss for the next value of `FORECAST_TRANSFORMER_CONFIG["target_column"]`. The conductivity-plug default uses Huber loss because it was more stable than plain MSE in local GPU experiments.

In [ ]:
if not FORECAST_TRANSFORMER_RUN:
    print("Good-data forecasting transformer fit skipped.")
else:
    torch.manual_seed(ML_STATE['SEED'])
    forecast_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if forecast_device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    class NextValueTransformer(nn.Module):
        def __init__(self, input_dim: int) -> None:
            super().__init__()
            # Project raw feature columns into the transformer embedding space.
            self.input_projection = nn.Linear(input_dim, FORECAST_TRANSFORMER_CONFIG["d_model"])
            # Learned positional embeddings let the transformer distinguish early vs late context rows.
            self.position_embedding = nn.Parameter(
                torch.zeros(1, FORECAST_TRANSFORMER_CONFIG["lookback"], FORECAST_TRANSFORMER_CONFIG["d_model"])
            )
            # Each encoder layer applies self-attention plus a feed-forward network.
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=FORECAST_TRANSFORMER_CONFIG["d_model"],
                nhead=FORECAST_TRANSFORMER_CONFIG["nhead"],
                dim_feedforward=FORECAST_TRANSFORMER_CONFIG["dim_feedforward"],
                dropout=FORECAST_TRANSFORMER_CONFIG["dropout"],
                activation="gelu",
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=FORECAST_TRANSFORMER_CONFIG["num_layers"])
            # The final context row summarises the history used to predict the next value.
            self.norm = nn.LayerNorm(FORECAST_TRANSFORMER_CONFIG["d_model"])
            self.head = nn.Linear(FORECAST_TRANSFORMER_CONFIG["d_model"], 1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.input_projection(x)
            x = x + self.position_embedding[:, : x.size(1)]
            x = self.encoder(x)
            x = self.norm(x[:, -1, :])
            return self.head(x).squeeze(-1)

    forecast_loader_kwargs = {}
    num_workers = int(FORECAST_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        forecast_loader_kwargs["num_workers"] = num_workers
        forecast_loader_kwargs["persistent_workers"] = bool(FORECAST_LOADER_CONFIG["persistent_workers"])
        forecast_loader_kwargs["prefetch_factor"] = int(FORECAST_LOADER_CONFIG["prefetch_factor"])
    if FORECAST_LOADER_CONFIG["pin_memory"] and forecast_device.type == "cuda":
        forecast_loader_kwargs["pin_memory"] = True
    forecast_non_blocking = bool(forecast_loader_kwargs.get("pin_memory", False) and forecast_device.type == "cuda")

    forecast_train_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_train_bundle["X_scaled"])),
        torch.from_numpy(forecast_train_bundle["y_scaled"]),
    )
    forecast_valid_good_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_valid_good_bundle["X_scaled"])),
        torch.from_numpy(forecast_valid_good_bundle["y_scaled"]),
    )
    forecast_valid_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_valid_bundle["X_scaled"])),
        torch.from_numpy(forecast_valid_bundle["y_scaled"]),
    )
    forecast_test_dataset = TensorDataset(
        torch.from_numpy(np.ascontiguousarray(forecast_test_bundle["X_scaled"])),
        torch.from_numpy(forecast_test_bundle["y_scaled"]),
    )
    forecast_train_loader = DataLoader(
        forecast_train_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=True,
        **forecast_loader_kwargs,
    )
    forecast_valid_good_loader = DataLoader(
        forecast_valid_good_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=False,
        **forecast_loader_kwargs,
    )
    forecast_valid_loader = DataLoader(
        forecast_valid_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=False,
        **forecast_loader_kwargs,
    )
    forecast_test_loader = DataLoader(
        forecast_test_dataset,
        batch_size=FORECAST_TRANSFORMER_CONFIG["batch_size"],
        shuffle=False,
        **forecast_loader_kwargs,
    )

    forecast_model = NextValueTransformer(input_dim=len(forecast_feature_columns)).to(forecast_device)
    # Huber is less sensitive to rare large errors; MSE penalizes large errors more strongly.
    forecast_loss_fn = nn.SmoothL1Loss() if FORECAST_TRANSFORMER_CONFIG["loss"] == "huber" else nn.MSELoss()
    forecast_optimizer = torch.optim.AdamW(
        forecast_model.parameters(),
        lr=FORECAST_TRANSFORMER_CONFIG["learning_rate"],
        weight_decay=FORECAST_TRANSFORMER_CONFIG["weight_decay"],
    )

    forecast_history = []
    forecast_best_state = None
    forecast_best_loss = np.inf
    forecast_best_epoch = 0
    forecast_patience_counter = 0

    def run_forecast_epoch(loader, training: bool) -> float:
        forecast_model.train(training)
        total_loss = 0.0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(forecast_device, non_blocking=forecast_non_blocking)
            batch_y = batch_y.to(forecast_device, non_blocking=forecast_non_blocking)
            with torch.set_grad_enabled(training):
                prediction = forecast_model(batch_x)
                loss = forecast_loss_fn(prediction, batch_y)
                if training:
                    forecast_optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if FORECAST_TRANSFORMER_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(forecast_model.parameters(), FORECAST_TRANSFORMER_CONFIG["gradient_clip_norm"])
                    forecast_optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
        return total_loss / max(len(loader.dataset), 1)

    for epoch in range(1, FORECAST_TRANSFORMER_CONFIG["epochs"] + 1):
        train_loss = run_forecast_epoch(forecast_train_loader, training=True)
        valid_loss = run_forecast_epoch(forecast_valid_good_loader, training=False)
        forecast_history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss})
        print(f"Forecast epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}")

        if valid_loss < forecast_best_loss - FORECAST_TRANSFORMER_CONFIG["min_delta"]:
            forecast_best_loss = valid_loss
            forecast_best_epoch = epoch
            forecast_patience_counter = 0
            forecast_best_state = copy.deepcopy(forecast_model.state_dict())
        else:
            forecast_patience_counter += 1
            if forecast_patience_counter >= FORECAST_TRANSFORMER_CONFIG["patience"]:
                print(f"Forecast early stopping triggered at epoch {epoch}")
                break

    if forecast_best_state is not None:
        forecast_model.load_state_dict(forecast_best_state)
    print({"forecast_best_epoch": forecast_best_epoch, "forecast_best_validation_loss": float(forecast_best_loss)})

#### Score Prediction Errors As Anomaly Signals

The anomaly score is the absolute prediction error after standardizing the target column. A high score means the next observed value was far from what the good-data model expected.

We set the threshold from the good validation rows only. Then we ask how often issue rows in validation and test exceed that threshold.

In [ ]:
if not FORECAST_TRANSFORMER_RUN:
    print("Good-data forecasting transformer scoring skipped.")
else:
    def score_forecast_loader(loader, metadata_frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
        forecast_model.eval()
        predictions = []
        targets = []
        with torch.no_grad():
            for batch_x, batch_y in loader:
                batch_x = batch_x.to(forecast_device, non_blocking=forecast_non_blocking)
                prediction = forecast_model(batch_x).detach().cpu().numpy()
                predictions.append(prediction)
                targets.append(batch_y.detach().cpu().numpy())
        predicted_scaled = np.concatenate(predictions) if predictions else np.array([], dtype=np.float32)
        target_scaled = np.concatenate(targets) if targets else np.array([], dtype=np.float32)
        result = metadata_frame.reset_index(drop=True).copy()
        result["split"] = split_name
        result["predicted_value"] = predicted_scaled * forecast_target_std + forecast_target_mean
        result["actual_value"] = target_scaled * forecast_target_std + forecast_target_mean
        result["forecast_error"] = result["actual_value"] - result["predicted_value"]
        result["forecast_abs_error_scaled"] = np.abs(target_scaled - predicted_scaled)
        return result

    forecast_valid_scores = score_forecast_loader(
        forecast_valid_loader,
        forecast_valid_bundle["metadata"],
        "validation",
    )
    forecast_test_scores = score_forecast_loader(
        forecast_test_loader,
        forecast_test_bundle["metadata"],
        "test",
    )

    threshold_source = forecast_valid_scores.loc[forecast_valid_scores["is_good"], "forecast_abs_error_scaled"]
    if threshold_source.empty:
        threshold_source = forecast_valid_scores["forecast_abs_error_scaled"]
    # The threshold is learned from good validation rows only, not from issue labels.
    forecast_error_threshold = float(
        threshold_source.quantile(float(FORECAST_TRANSFORMER_CONFIG["threshold_quantile"]))
    )

    forecast_score_rows = []
    for split_name, score_frame in [("validation", forecast_valid_scores), ("test", forecast_test_scores)]:
        # Prediction errors above the threshold are treated as anomalies.
        score_frame["predicted_anomaly"] = score_frame["forecast_abs_error_scaled"] >= forecast_error_threshold
        y_true_anomaly = score_frame["is_issue"].astype(int)
        y_pred_anomaly = score_frame["predicted_anomaly"].astype(int)
        forecast_score_rows.append(
            {
                "split": split_name,
                "rows": len(score_frame),
                "issue_rows": int(y_true_anomaly.sum()),
                "predicted_anomaly_rows": int(y_pred_anomaly.sum()),
                "median_abs_error_scaled": float(score_frame["forecast_abs_error_scaled"].median()),
                "p95_abs_error_scaled": float(score_frame["forecast_abs_error_scaled"].quantile(0.95)),
                "forecast_anomaly_f1": float(f1_score(y_true_anomaly, y_pred_anomaly, zero_division=0)),
            }
        )
        print(f"\n{split_name.title()} forecast-error anomaly report")
        print(
            classification_report(
                y_true_anomaly,
                y_pred_anomaly,
                labels=[0, 1],
                target_names=["not issue", "issue"],
                zero_division=0,
            )
        )

    forecast_score_summary = pd.DataFrame(forecast_score_rows).set_index("split")
    display(forecast_score_summary)
    print(
        {
            "threshold_quantile": FORECAST_TRANSFORMER_CONFIG["threshold_quantile"],
            "forecast_error_threshold": forecast_error_threshold,
            "threshold_rows_from_good_validation": int(len(threshold_source)),
        }
    )

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8), sharey=True)
    for axis, (split_name, score_frame) in zip(axes, [("validation", forecast_valid_scores), ("test", forecast_test_scores)]):
        good_scores = score_frame.loc[score_frame["is_good"], "forecast_abs_error_scaled"]
        issue_scores = score_frame.loc[score_frame["is_issue"], "forecast_abs_error_scaled"]
        bins = np.linspace(
            0,
            max(float(score_frame["forecast_abs_error_scaled"].quantile(0.995)), forecast_error_threshold, 1e-6),
            45,
        )
        axis.hist(good_scores, bins=bins, alpha=0.65, label="good next rows", color="#1f77b4")
        axis.hist(issue_scores, bins=bins, alpha=0.65, label="issue next rows", color="#d62728")
        axis.axvline(forecast_error_threshold, color="#111827", linestyle="--", linewidth=1.8, label="threshold")
        axis.set_title(f"{split_name.title()} next-value forecast errors")
        axis.set_xlabel("Absolute standardised prediction error")
        axis.grid(axis="y", alpha=0.2)
    axes[0].set_ylabel("Window count")
    axes[1].legend(bbox_to_anchor=(1.02, 1.0), loc="upper left")
    plt.tight_layout()
    plt.show()

#### Reflection Questions: Forecast-Error Transformer

1. Are large prediction errors lining up with known issue flags, or mostly with legitimate rapid changes?
2. Would this detector work better if it predicted several future steps instead of only the next value?
3. Should the anomaly score combine several measurement columns, or is one operationally important column enough?
4. Does training only on good rows make the detector more sensitive to novel failure modes, or too sensitive to normal regime changes?

## Part 5 — Reflection And Next Steps

### Wrap-Up And Questions To Explore

This notebook moved from "train a model" to "decide what is worth changing". Questions to carry forward:

1. Did hyperparameter tuning change the test result as much as the feature or target changes did, for both the Random Forest and the CNN?
Hint: compare the search tables with the offline tree and CNN study summaries.

2. Which context features would you add next?
Hint: think about lags, rolling statistics, slopes, or relationships between sensors.

3. Does the target itself need to change?
Hint: a collapsed `1 / 34 / 9` target or a binary good-versus-issue target can be closer to the operational decision. For oxygen, ask whether `1/2` and `3/4` belong together rather than reusing the CTD collapse blindly.

4. In the loss comparison, which option gave the best balance between recall and precision for the rarest class, and would that choice change for a different dataset preset?

5. Where would a forecasting detector fit operationally: as a replacement for supervised QC models, or as a second opinion that catches failure modes the labels never covered?

6. Could even more temporal context help?
Hint: longer windows, TCNs, GRUs/LSTMs, or hierarchical windows.